In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer360_master

In [0]:
select distinct a.BILLING_NPI, b.NPI from com_edp_prd.com_raw.kom_medical_events as a
left join com_edp_prd.com_raw.kom_providers b on a.BILLING_NPI = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL' 
where b.npi is not null

In [0]:
select distinct a.BILLING_NPI, b.NPI 
from com_edp_prd.com_raw.kom_medical_events as a
left join com_edp_prd.com_raw.kom_providers b on a.BILLING_NPI = b.NPI and b.PROVIDER_TYPE = 'ORGANIZATION' 
where b.npi is not null

In [0]:
select distinct a.PHARMACY_NPI, b.NPI from com_edp_prd.com_raw.kom_pharmacy_events as a
left join com_edp_prd.com_raw.kom_providers b on a.PHARMACY_NPI = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL' 
where b.npi is not null

In [0]:
select distinct a.PHARMACY_NPI, b.NPI from com_edp_prd.com_raw.kom_pharmacy_events as a
left join com_edp_prd.com_raw.kom_providers b on a.PHARMACY_NPI = b.NPI and b.PROVIDER_TYPE = 'ORGANIZATION' 
where b.npi is not null

In [0]:
select age_bucket, count(distinct PATIENT_ID)
from cmpa_insights_internal_schema.patient360_master
group by 1 order by 2 desc

### Patient 360

In [0]:
select * from cmpa_insights_internal_schema.patient360_master

In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ----------------------------------------------------------
-- Claims universes (5y, 3y) Dx + Tx, with NPIs
-- ----------------------------------------------------------
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* =========================================================
   all_tx_claims_5yr carries TX_CODE (date-bounded)
   ========================================================= */
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* =========================================================
   NEW: all_tx_claims_alltime (NO date restriction)
   - same tx definition as all_tx_claims_5yr
   - used ONLY for first_tx_after_diagnosis
   ========================================================= */
all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* =========================================================
   all_claims_5yr aligned schema (adds TX_CODE for dx as NULL)
   ========================================================= */
all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE
    FROM all_tx_claims_5yr
),

all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
)

select *
from all_claims_5yr
where patient_id in ('R880T4GS')

In [0]:
-- =============================================================================
-- Patient HCP Visit Summary View - Top 5 based on 3-Year activity
--   + Adds: first_tx_after_diagnosis, time_dx_to_first_tx_in_months,
--           treatment_period_months, elaprase_fills
--
-- CHANGE (per request):
--   - first_tx_after_diagnosis should be ALL-TIME (no date restriction),
--     with only constraint: tx.fill_date >= incidence_date
--   - Implemented via new CTE: all_tx_claims_alltime (same tx definition,
--     no date filters), and first_tx_after_diagnosis now uses it.
-- =============================================================================

CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ----------------------------------------------------------
-- Claims universes (5y, 3y) Dx + Tx, with NPIs
-- ----------------------------------------------------------
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* =========================================================
   all_tx_claims_5yr carries TX_CODE (date-bounded)
   ========================================================= */
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* =========================================================
   NEW: all_tx_claims_alltime (NO date restriction)
   - same tx definition as all_tx_claims_5yr
   - used ONLY for first_tx_after_diagnosis
   ========================================================= */
all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* =========================================================
   all_claims_5yr aligned schema (adds TX_CODE for dx as NULL)
   ========================================================= */
all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE
    FROM all_tx_claims_5yr
),

all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- ----------------------------------------------------------
-- First Dx / First Tx (5y)
-- ----------------------------------------------------------
first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),
provider_dim AS (
    SELECT
        npi,
        CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
        primary_specialty
    FROM com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
),
first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),

first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),
first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),
first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

-- ----------------------------------------------------------
-- Most-seen ranking (Top 5 by 3y) with 5y counts + 5y last-visit
-- ----------------------------------------------------------
most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),

-- ----------------------------------------------------------
-- Historical First Dates (No Date Restrictions)
-- ----------------------------------------------------------
historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),
historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),

-- ----------------------------------------------------------
-- Latest Claim HCP (5y universe)  [HCP attribution only when NPI exists]
-- ----------------------------------------------------------
latest_claim_hcp_ranked AS (
    SELECT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY FILL_DATE DESC, NPI ASC
        ) AS rn
    FROM all_claims_5yr
    WHERE NPI IS NOT NULL
),
latest_claim_hcp AS (
    SELECT
        PATIENT_ID,
        NPI AS latest_claim_hcp_npi,
        FILL_DATE AS latest_claim_date
    FROM latest_claim_hcp_ranked
    WHERE rn = 1
),
latest_claim_hcp_visit_count_5yr AS (
    SELECT
        lch.PATIENT_ID,
        lch.latest_claim_hcp_npi,
        COUNT(DISTINCT ac.FILL_DATE) AS latest_claim_hcp_visit_count_5yr
    FROM latest_claim_hcp lch
    LEFT JOIN all_claims_5yr ac
      ON lch.PATIENT_ID = ac.PATIENT_ID
     AND lch.latest_claim_hcp_npi = ac.NPI
    GROUP BY lch.PATIENT_ID, lch.latest_claim_hcp_npi
),

-- ----------------------------------------------------------
-- Latest Treatment HCP (5y universe) [HCP attribution only when NPI exists]
-- ----------------------------------------------------------
most_recent_tx_hcp_ranked AS (
    SELECT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY FILL_DATE DESC, NPI ASC
        ) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
),
most_recent_tx_hcp AS (
    SELECT
        PATIENT_ID,
        NPI AS latest_treatment_hcp_npi,
        FILL_DATE AS latest_treatment_date
    FROM most_recent_tx_hcp_ranked
    WHERE rn = 1
),
latest_treatment_hcp_visit_count AS (
    SELECT
        mrt.PATIENT_ID,
        mrt.latest_treatment_hcp_npi,
        COUNT(DISTINCT ac.FILL_DATE) AS latest_treatment_hcp_visit_count_5yr
    FROM most_recent_tx_hcp mrt
    LEFT JOIN all_claims_5yr ac
      ON mrt.PATIENT_ID = ac.PATIENT_ID
     AND mrt.latest_treatment_hcp_npi = ac.NPI
    GROUP BY mrt.PATIENT_ID, mrt.latest_treatment_hcp_npi
),

-- ----------------------------------------------------------
-- Patient-level Latest Dates (ignore NPI so dates never go NULL when claims exist)
-- ----------------------------------------------------------
latest_claim_date_patient AS (
  SELECT
    PATIENT_ID,
    MAX(FILL_DATE) AS latest_claim_date_any
  FROM all_claims_5yr
  GROUP BY PATIENT_ID
),
latest_treatment_date_patient AS (
  SELECT
    PATIENT_ID,
    MAX(FILL_DATE) AS latest_treatment_date_any
  FROM all_tx_claims_5yr
  GROUP BY PATIENT_ID
),

-- ----------------------------------------------------------
-- latest_mpsii_tx_type derived ONLY from all_tx_claims_5yr (windowed)
-- ----------------------------------------------------------
latest_mpsii_treatment_type AS (
  SELECT
    patient_id,
    CASE
      WHEN tx_code IN ('54092070001','540920700','J1743') THEN 'Elaprase'
      ELSE 'Other ERT Proc'
    END AS latest_mpsii_tx_type
  FROM (
    SELECT
      patient_id,
      fill_date,
      tx_code,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC, tx_code ASC
      ) AS rn
    FROM all_tx_claims_5yr
    WHERE tx_code IS NOT NULL
      AND fill_date BETWEEN '2023-08-01' AND '${end_date}'
  )
  WHERE rn = 1
),

-- ----------------------------------------------------------
-- First Treatment After Diagnosis (ALL-TIME tx; must be after dx date)
-- ----------------------------------------------------------
first_tx_after_diagnosis AS (
  SELECT
    tx.patient_id,
    MIN(tx.fill_date) AS first_tx_after_diagnosis
  FROM all_tx_claims_alltime tx
  INNER JOIN historical_first_dx dx
    ON tx.patient_id = dx.patient_id
  WHERE tx.fill_date >= dx.incidence_date
  GROUP BY tx.patient_id
),

-- ----------------------------------------------------------
-- Elaprase fills (distinct tx dates within window) using all_tx_claims_5yr
-- NOTE: If you want true Elaprase-only fills, add tx_code filter here.
-- ----------------------------------------------------------
elaprase_fills AS (
  SELECT
    patient_id,
    COUNT(DISTINCT fill_date) AS elaprase_fills
  FROM all_tx_claims_5yr
  WHERE fill_date BETWEEN '2023-08-01' AND '${end_date}'
    AND tx_code IN ('54092070001','540920700','J1743')
  GROUP BY patient_id
),

-- ----------------------------------------------------------
-- Patient Level Information
-- ----------------------------------------------------------
patient_demographics AS (
    SELECT *
    FROM (
        SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
               ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
    WHERE rn = 1
),
patient_geography AS (
    SELECT patient_id, patient_state
    FROM (
        SELECT
            PATIENT_ID,
            patient_state,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY
                    CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
                    VALID_TO_DATE DESC
            ) AS rn
        FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
    )
    WHERE rn = 1
),

/* =========================================================
   Pivot Top-5 Most-seen (avoid 10+ joins)
   ========================================================= */
most_seen_pivot AS (
    SELECT
        PATIENT_ID,

        MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
        MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
        MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

        MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
        MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
        MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

        MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
        MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
        MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

        MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
        MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
        MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

        MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
        MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
        MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

    FROM most_seen_combined_stats
    GROUP BY PATIENT_ID
)

SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,

    -- Historical First Dates
    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,

    -- Latest Claim date: always present if any claim exists
    COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,

    -- Latest Claim (HCP attribution only when NPI exists)
    lch.latest_claim_hcp_npi,
    pdlch.provider_name     AS latest_claim_hcp_name,
    pdlch.primary_specialty AS latest_claim_hcp_specialty,
    COALESCE(lchvc.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

    -- Latest Claim HCO details
    ref1.hco_npi  AS latest_claim_hcp_hco_npi,
    ref1.hco_name AS latest_claim_hcp_hco_name,

    -- Latest Treatment date: always present if any tx claim exists
    COALESCE(mrt.latest_treatment_date, ltd.latest_treatment_date_any) AS latest_treatment_date,

    -- Latest treatment type (windowed)
    lmt.latest_mpsii_tx_type,

    -- First treatment after diagnosis (ALL-TIME)
    fta.first_tx_after_diagnosis,

    -- time dx -> first tx (months)
    ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
      AS time_dx_to_first_tx_in_months,

    -- treatment period (months): first tx -> latest tx (patient-level latest)
    ROUND(
      MONTHS_BETWEEN(
        COALESCE(mrt.latest_treatment_date, ltd.latest_treatment_date_any),
        fta.first_tx_after_diagnosis
      ),
      0
    ) AS treatment_period_months,

    -- Elaprase fills (windowed, Elaprase-only)
    COALESCE(ef.elaprase_fills, 0) AS elaprase_fills,

    -- Latest Treatment (HCP attribution only when NPI exists)
    mrt.latest_treatment_hcp_npi,
    pdtch.provider_name     AS latest_treatment_hcp_name,
    pdtch.primary_specialty AS latest_treatment_hcp_specialty,
    COALESCE(lthvc.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,

    -- Latest Treatment HCO details
    ref2.hco_npi  AS latest_treatment_hcp_hco_npi,
    ref2.hco_name AS latest_treatment_hcp_hco_name,

    -- First Dx HCP (5y stats)
    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

    -- First Tx HCP (5y stats)
    fth.first_tx_hcp AS first_tx_hcp_5yr,
    COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
    COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
    fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

    -- Top 5 most-seen (3y ranked, 5y counts + 5y last visit)
    msp.most_seen_hcp1_3yr_ranked,
    COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
    msp.most_seen_hcp1_last_visit_5yr,

    msp.most_seen_hcp2_3yr_ranked,
    COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
    msp.most_seen_hcp2_last_visit_5yr,

    msp.most_seen_hcp3_3yr_ranked,
    COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
    msp.most_seen_hcp3_last_visit_5yr,

    msp.most_seen_hcp4_3yr_ranked,
    COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
    msp.most_seen_hcp4_last_visit_5yr,

    msp.most_seen_hcp5_3yr_ranked,
    COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
    msp.most_seen_hcp5_last_visit_5yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd
    ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg
    ON ep.PATIENT_ID = pg.PATIENT_ID
LEFT JOIN historical_first_dx hfdx
    ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx
    ON ep.PATIENT_ID = hftx.PATIENT_ID

-- First tx after dx (ALL-TIME)
LEFT JOIN first_tx_after_diagnosis fta
    ON ep.PATIENT_ID = fta.PATIENT_ID

-- Elaprase fills (windowed)
LEFT JOIN elaprase_fills ef
    ON ep.PATIENT_ID = ef.PATIENT_ID

-- Patient-level latest dates
LEFT JOIN latest_claim_date_patient lcd
    ON ep.PATIENT_ID = lcd.PATIENT_ID
LEFT JOIN latest_treatment_date_patient ltd
    ON ep.PATIENT_ID = ltd.PATIENT_ID

-- Latest tx type (windowed)
LEFT JOIN latest_mpsii_treatment_type lmt
    ON ep.PATIENT_ID = lmt.PATIENT_ID

-- Latest claim HCP joins
LEFT JOIN latest_claim_hcp lch
    ON ep.PATIENT_ID = lch.PATIENT_ID
LEFT JOIN latest_claim_hcp_visit_count_5yr lchvc
    ON ep.PATIENT_ID = lchvc.PATIENT_ID
   AND lch.latest_claim_hcp_npi = lchvc.latest_claim_hcp_npi
LEFT JOIN provider_dim pdlch
    ON lch.latest_claim_hcp_npi = pdlch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 ref1
    ON lch.latest_claim_hcp_npi = ref1.hcp_npi

-- Latest treatment HCP joins
LEFT JOIN most_recent_tx_hcp mrt
    ON ep.PATIENT_ID = mrt.PATIENT_ID
LEFT JOIN latest_treatment_hcp_visit_count lthvc
    ON ep.PATIENT_ID = lthvc.PATIENT_ID
   AND mrt.latest_treatment_hcp_npi = lthvc.latest_treatment_hcp_npi
LEFT JOIN provider_dim pdtch
    ON mrt.latest_treatment_hcp_npi = pdtch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 ref2
    ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

-- First Dx/Tx joins
LEFT JOIN first_dx_hcp fdh
    ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs
    ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_tx_hcp fth
    ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths
    ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
    ON ep.PATIENT_ID = fthtx.PATIENT_ID

-- Pivoted top-5 most-seen
LEFT JOIN most_seen_pivot msp
    ON ep.PATIENT_ID = msp.PATIENT_ID

ORDER BY ep.PATIENT_ID;

-- Create the table from the temp view
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_base AS
SELECT DISTINCT * FROM patient_hcp_visit_summary;


### Marketing HCPs

In [0]:
select distinct npi
from com_raw.kom_providers
-- where 
-- PROVIDER_TYPE = 'INDIVIDUAL'
-- npi in ('1295888642'
-- '1609881366'
-- '1588984942'
-- '1447357405'
-- '1841267051'
-- '1235197864'
-- '1225027782'
-- '1922138437'
-- '1982845327'
-- '1952341885'
-- '1740350024'
-- '1215958756'
-- '1043234461'
-- '1417272188'
-- '1144269341'
-- '1063615672'
-- '1689680613'
-- '1447576434'
-- '1881885531'
-- '1699069583'
-- '1487808275'
-- '1235450693'
-- '1023215217'
-- '1386103216'
-- '1649297573'
-- '1932119377'
-- '1003849134'
-- '1528395985'
-- '1477998730'
-- '1568538858'
-- '1154335719'
-- '1073590550'
-- '1538723002'
-- '1659343341'
-- '1487677340'
-- '1407113178'
-- '1346230398'
-- '1841367604'
-- '1851703284'
-- '1417930678'
-- '1275535395'
-- '1093001448'
-- '1982993390'
-- '1174525711'
-- '1831116656'
-- '1659387272'
-- '1215314265'
-- '1841270451'
-- '1780970020'
-- '1477904068'
-- '1790238921'
-- '1013087998'
-- '1194986430'
-- '1699830547'
-- '1881032050'
-- '1073717948'
-- '1942444617'
-- '1376610139'
-- '1750373353'
-- '1215044557'
-- '1598185126'
-- '1447678230'
-- '1629056536'
-- '1740691427'
-- '1982687422'
-- '1568692812'
-- '1194785196'
-- '1750780011'
-- '1811009939'
-- '1679867808'
-- '1346528072'
-- '1487996203'
-- '1669762746'
-- '1003059932'
-- '1316104508'
-- '1447259494'
-- '1740291848'
-- '1407946957'
-- '1568412005'
-- '1427347962'
-- '1881217453'
-- '1780025585'
-- '1649405499'
-- '1164623682'
-- '1760471569'
-- '1881623239'
-- '1083709471'
-- '1457592479'
-- '1396156931'
-- '1659535839'
-- '1043200173'
-- '1629336276'
-- '1871556878'
-- '1376520056'
-- '1477643856'
-- '1851787626'
-- '1952795403'
-- '1154416675'
-- '1427685916'
-- '1609040716'
-- '1447287933'
-- '1750675161'
-- '1114964228'
-- '1699850057'
-- '1174884225'
-- '1710302880'
-- '1871629246'
-- '1174524011'
-- '1336170059'
-- '1023379336'
-- '1861753303'
-- '1679597983'
-- '1578055935'
-- '1588989784'
-- '1043331044'
-- '1588714463'
-- '1174713168'
-- '1669188645'
-- '1104888098'
-- '1407066384'
-- '1609408210'
-- '1912086760'
-- '1679356372'
-- '1235949199'
-- '1730322462'
-- '1548645237'
-- '1073909933'
-- '1144286980'
-- '1194213678'
-- '1265922702'
-- '1346526324'
-- '1043491483'
-- '1760547418'
-- '1114148020'
-- '1255416012'
-- '1215989132'
-- '1558387308'
-- '1679586275'
-- '1609887983'
-- '1497011225'
-- '1801082896'
-- '1922458736'
-- '1972861441'
-- '1588005870'
-- '1447270053'
-- '1467896464'
-- '1548666258'
-- '1538175286'
-- '1184687162'
-- '1710171715'
-- '1487734091'
-- '1366093023'
-- '1710065271'
-- '1265463772'
-- '1851313696'
-- '1104138676'
-- '1023404597'
-- '1093714040'
-- '1669976098'
-- '1881649796'
-- '1477146298'
-- '1619065190'
-- '1902969546'
-- '1992060958'
-- '1144261595'
-- '1558339358'
-- '1831157916'
-- '1508413147'
-- '1497969786'
-- '1063153633'
-- '1619635471'
-- '1659094928'
-- '1861673972'
-- '1962629600'
-- '1699718031'
-- '1770891020'
-- '1134443617'
-- '1184642951'
-- '1235150491'
-- '1255758637'
-- '1568469245'
-- '1205958964'
-- '1316151814'
-- '1942460837'
-- '1215961347'
-- '1508182049'
-- '1306016860'
-- '1336572544'
-- '1144715251'
-- '1346212966'
-- '1881907533'
-- '1568562445'
-- '1538396007'
-- '1265405120'
-- '1245471234'
-- '1932177151'
-- '1902260722'
-- '1932117470'
-- '1548266299'
-- '1780817825'
-- '1295930212'
-- '1376575175'
-- '1891085528'
-- '1245625078'
-- '1912085788'
-- '1124187018'
-- '1174525059'
-- '1255400891'
-- '1760464044'
-- '1982281432'
-- '1447297551'
-- '1093039034'
-- '1235514688'
-- '1639339450'
-- '1770719841'
-- '1922051762'
-- '1528038114'
-- '1578939864'
-- '1346482361'
-- '1124694468'
-- '1275882599'
-- '1225281470'
-- '1235623018'
-- '1942628912'
-- '1275964504'
-- '1366083040'
-- '1184656613'
-- '1194878512'
-- '1952620791'
-- '1669453148'
-- '1881230811'
-- '1477865731'
-- '1912936725'
-- '1851785836'
-- '1598955973'
-- '1841456555'
-- '1306525597'
-- '1689002628'
-- '1730508649'
-- '1932573912'
-- '1043240500'
-- '1689988446'
-- '1376705335'
-- '1821000159'
-- '1962460741'
-- '1669456133'
-- '1255576542'
-- '1972000529'
-- '1215939863'
-- '1528267523'
-- '1932337144'
-- '1952475279'
-- '1144253154'
-- '1598928103'
-- '1700267978'
-- '1952313298'
-- '1336298405'
-- '1982661997'
-- '1265694244'
-- '1144375965'
-- '1164695730'
-- '1407995400'
-- '1578019238'
-- '1124224936'
-- '1619302338'
-- '1164408274'
-- '1780656173'
-- '1346465630'
-- '1770711269'
-- '1629178306'
-- '1649776618'
-- '1518420645'
-- '1366490492'
-- '1891871877'
-- '1518059369'
-- '1770639361'
-- '1477537405'
-- '1346512167'
-- '1245317189'
-- '1851393474'
-- '1720620198'
-- '1083690572'
-- '1710242730'
-- '1699933903'
-- '1780624577'
-- '1003858036'
-- '1902403843'
-- '1083929269'
-- '1407464233'
-- '1154835593'
-- '1053532366'
-- '1225026867'
-- '1174753529'
-- '1598376949'
-- '1104935840'
-- '1710961602'
-- '1336149392'
-- '1770564825'
-- '1922271188'
-- '1669459798'
-- '1598779811'
-- '1154465672'
-- '1083946685'
-- '1770678930'
-- '1164989497'
-- '1912258302'
-- '1114136934'
-- '1205213469'
-- '1245438167'
-- '1336109917'
-- '1427311927'
-- '1710203518'
-- '1730457789'
-- '1326079013'
-- '1639322407'
-- '1104389394'
-- '1275792178'
-- '1154639623'
-- '1245410844'
-- '1639221021'
-- '1134516982'
-- '1912226721'
-- '1427264969'
-- '1649765322'
-- '1851435408'
-- '1326083312'
-- '1427391705'
-- '1326529132'
-- '1730250796'
-- '1326492851'
-- '1548289093'
-- '1528355997'
-- '1841393394'
-- '1144217886'
-- '1871976639'
-- '1841480217'
-- '1033489794'
-- '1518192616'
-- '1578542130'
-- '1710440086'
-- '1972029544'
-- '1376868158'
-- '1518634427'
-- '1992899413'
-- '1457951485'
-- '1487128989'
-- '1710446190'
-- '1316520422'
-- '1831545326'
-- '1649415837'
-- '1376769943'
-- '1245226059'
-- '1467048520'
-- '1902446081'
-- '1720207996'
-- '1477566222'
-- '1609890334'
-- '1841233798'
-- '1093896326'
-- '1396781696'
-- '1457675415'
-- '1538629894'
-- '1396476685'
-- '1750481800'
-- '1376745018'
-- '1982835518'
-- '1598090326'
-- '1932865177'
-- '1457910424'
-- '1871927111'
-- '1124374137'
-- '1679523435'
-- '1235211848'
-- '1023246600'
-- '1255112389'
-- '1053516583'
-- '1154421642'
-- '1003067331'
-- '1467447748'
-- '1316144645'
-- '1376522912'
-- '1831658855'
-- '1346835162'
-- '1386802387'
-- '1437322310'
-- '1588143887'
-- '1609074301'
-- '1669694857'
-- '1275778136'
-- '1972759629'
-- '1477755155'
-- '1821297490'
-- '1841267267'
-- '1831283837'
-- '1548406010'
-- '1790709319'
-- '1467125260'
-- '1194912592'
-- '1972958122'
-- '1235332701'
-- '1316152127'
-- '1699965186'
-- '1861107716'
-- '1376572107'
-- '1821178237'
-- '1043273766'
-- '1740508928'
-- '1093391260'
-- '1023196896'
-- '1295397933'
-- '1124288907'
-- '1922011030'
-- '1639288723'
-- '1548234784'
-- '1841243755'
-- '1811901424'
-- '1609058338'
-- '1619929080'
-- '1215164074'
-- '1083909733'
-- '1518018076'
-- '1902901242'
-- '1003121799'
-- '1003251943'
-- '1235196106'
-- '1689635047'
-- '1629049853'
-- '1467553131'
-- '1750792792'
-- '1174768030'
-- '1184045148'
-- '1174593735'
-- '1386656767'
-- '1629369798'
-- '1780146266'
-- '1205938156'
-- '1568429199'
-- '1659472314'
-- '1982044012'
-- '1346332780'
-- '1740442268'
-- '1992810972'
-- '1619406824'
-- '1518131838'
-- '1013987262'
-- '1033742739'
-- '1407891963'
-- '1134170939'
-- '1144386087'
-- '1013013424'
-- '1295797330'
-- '1346329810'
-- '1538461728'
-- '1689052300'
-- '1023105301'
-- '1942240080'
-- '1497491021'
-- '1952035172'
-- '1255075032'
-- '1700142155'
-- '1669714275'
-- '1942594080'
-- '1376709519'
-- '1699781609'
-- '1205061389'
-- '1720073992'
-- '1992098503'
-- '1265597124'
-- '1639175649'
-- '1942364690'
-- '1760519367'
-- '1982927976'
-- '1033479720'
-- '1346653284'
-- '1861036139'
-- '1922272954'
-- '1225557762'
-- '1235870247'
-- '1306454582'
-- '1619353703'
-- '1154705606'
-- '1215227293'
-- '1699847426'
-- '1902361595'
-- '1750360764'
-- '1982807053'
-- '1558750596'
-- '1740770809'
-- '1841365822'
-- '1205345741'
-- '1437173085'
-- '1558762203'
-- '1669614236'
-- '1366768640'
-- '1922086404'
-- '1386616993'
-- '1467192310'
-- '1558334862'
-- '1295054542'
-- '1396710588'
-- '1457329773'
-- '1881122018'
-- '1033108980'
-- '1467452276'
-- '1699295493'
-- '1417047390'
-- '1568455533'
-- '1962467878'
-- '1770873978'
-- '1386051464'
-- '1457302879'
-- '1801818653'
-- '1407836299'
-- '1467480814'
-- '1871578666'
-- '1659465987'
-- '1033463054'
-- '1932357217'
-- '1598175127'
-- '1881734283'
-- '1629495023'
-- '1831156728'
-- '1841679198'
-- '1124029756'
-- '1730157686'
-- '1750584199'
-- '1780826271'
-- '1730613811'
-- '1164015947'
-- '1568463537'
-- '1437449311'
-- '1124221981'
-- '1578656658'
-- '1942236492'
-- '1265499214'
-- '1346407558'
-- '1710019369'
-- '1821061060'
-- '1437477411'
-- '1720582422'
-- '1992743546'
-- '1023487261'
-- '1528019684'
-- '1508451188'
-- '1841367752'
-- '1861629321'
-- '1699758433'
-- '1184826273'
-- '1659515567'
-- '1720735913'
-- '1912190547'
-- '1023006798'
-- '1073615928'
-- '1134176183'
-- '1942308192'
-- '1841213972'
-- '1154841617'
-- '1467413658'
-- '1639114424'
-- '1407822539'
-- '1487066791'
-- '1376628156'
-- '1295824787'
-- '1396037750'
-- '1013996214'
-- '1336618289'
-- '1376511840'
-- '1952445900'
-- '1134337512'
-- '1619985462'
-- '1407417520'
-- '1033538699'
-- '1205421633'
-- '1114019080'
-- '1346216652'
-- '1821258583'
-- '1912114034'
-- '1265415913'
-- '1871555805'
-- '1225352297'
-- '1326206335'
-- '1548239577'
-- '1609882703'
-- '1073600177'
-- '1619997830'
-- '1184088676'
-- '1750677894'
-- '1790103067'
-- '1851553085'
-- '1598379919'
-- '1154413037'
-- '1487090601'
-- '1861811242'
-- '1003831512'
-- '1134447444'
-- '1235695818'
-- '1558685578'
-- '1831113919'
-- '1366774119'
-- '1740495753'
-- '1902972284'
-- '1467023176'
-- '1487752655'
-- '1255773966'
-- '1578616207'
-- '1720355928'
-- '1093898447'
-- '1447273362'
-- '1548478977'
-- '1629020482'
-- '1366888984'
-- '1053418145'
-- '1144580424'
-- '1407833353'
-- '1053907709'
-- '1184159139'
-- '1679552392'
-- '1174545644'
-- '1538166129'
-- '1558311431'
-- '1891307302'
-- '1750024089'
-- '1134129695'
-- '1972389831'
-- '1841884079'
-- '1689692931'
-- '1043426539'
-- '1538795406'
-- '1679777528'
-- '1689616997'
-- '1013972231'
-- '1437123619'
-- '1467688523'
-- '1891966560'
-- '1538320916'
-- '1821418674'
-- '1710011390'
-- '1841238011'
-- '1891421277'
-- '1215407093'
-- '1912401670'
-- '1831385053'
-- '1396748950'
-- '1457462426'
-- '1194373498'
-- '1275583494'
-- '1669960225'
-- '1730371030'
-- '1346799376'
-- '1437214608'
-- '1942533443'
-- '1174916472'
-- '1275709644'
-- '1245348374'
-- '1508386947'
-- '1295889905'
-- '1811969595'
-- '1003084377'
-- '1023176880'
-- '1255581070'
-- '1699209155'
-- '1245506914'
-- '1598722233'
-- '1609910033'
-- '1558654392'
-- '1689688681'
-- '1972560464'
-- '1689328551'
-- '1831292036'
-- '1265410708'
-- '1235405986'
-- '1255716320'
-- '1487038139'
-- '1336137009'
-- '1114390663'
-- '1700157708'
-- '1114901337'
-- '1265423941'
-- '1184702409'
-- '1124447313'
-- '1568719292'
-- '1316113905'
-- '1437159365'
-- '1740443332'
-- '1306360540'
-- '1447519921'
-- '1326138983'
-- '1629249859'
-- '1164482386'
-- '1053349159'
-- '1487602728'
-- '1346237195'
-- '1386645125'
-- '1669538815'
-- '1578970349'
-- '1659517977'
-- '1245641620'
-- '1790896215'
-- '1124380258'
-- '1598702094'
-- '1053483792'
-- '1992880215'
-- '1518928456'
-- '1639597396'
-- '1659540375'
-- '1669687596'
-- '1962675264'
-- '1649372079'
-- '1992860621'
-- '1992142624'
-- '1295052637'
-- '1164800264'
-- '1306130786'
-- '1891780342'
-- '1740440916'
-- '1093041808'
-- '1336376318'
-- '1922273580'
-- '1366492779'
-- '1114997350'
-- '1538598701'
-- '1235346198'
-- '1952394520'
-- '1497744817'
-- '1114001856'
-- '1730295478'
-- '1942864574'
-- '1376833848'
-- '1376793331'
-- '1518227305'
-- '1831360304'
-- '1063065506'
-- '1659931996'
-- '1982709564'
-- '1114942265'
-- '1750591004'
-- '1144220088'
-- '1902122575'
-- '1275598047'
-- '1588771141'
-- '1720382054'
-- '1154387801'
-- '1376592865'
-- '1932424231'
-- '1801207543'
-- '1003201658'
-- '1164429114'
-- '1629090022'
-- '1386176303'
-- '1285807990'
-- '1831391218'
-- '1922073451'
-- '1871597815'
-- '1891988598'
-- '1104970359'
-- '1134232598'
-- '1912966888'
-- '1356575088'
-- '1013952142'
-- '1811309248'
-- '1366489650'
-- '1194106534'
-- '1902217748'
-- '1053673392'
-- '1366407926'
-- '1437131323'
-- '1154392462'
-- '1356792121'
-- '1538239553'
-- '1831405166'
-- '1386646735'
-- '1891784617'
-- '1730159906'
-- '1790042851'
-- '1760503205'
-- '1679525125'
-- '1891708368'
-- '1104084516'
-- '1407975469'
-- '1154620417'
-- '1457763583'
-- '1023044856'
-- '1174828065'
-- '1699754994'
-- '1699033860'
-- '1083843577'
-- '1629335849'
-- '1871560409'
-- '1619921442'
-- '1417158130'
-- '1578876249'
-- '1689619033'
-- '1033143441'
-- '1013918341'
-- '1760860753'
-- '1306100623'
-- '1740227388'
-- '1073999843'
-- '1922119957'
-- '1932527116'
-- '1215298906'
-- '1801231709'
-- '1710281209'
-- '1174526263'
-- '1154580173'
-- '1801983796'
-- '1912260985'
-- '1457677981'
-- '1730289240'
-- '1487944799'
-- '1760748149'
-- '1770894438'
-- '1427112416'
-- '1588679039'
-- '1356306427'
-- '1669892642'
-- '1891774345'
-- '1174504096'
-- '1851475271'
-- '1164661922'
-- '1366468241'
-- '1346205721'
-- '1417333139'
-- '1770565798'
-- '1619980547'
-- '1992790760'
-- '1467849356'
-- '1558470310'
-- '1659342889'
-- '1215039227'
-- '1871564682'
-- '1124059597'
-- '1467689349'
-- '1265550677'
-- '1467417410'
-- '1457593667'
-- '1528170685'
-- '1871564443'
-- '1942578208'
-- '1427410885'
-- '1083642375'
-- '1174562417'
-- '1588680094'
-- '1992188601'
-- '1568419471'
-- '1730274796'
-- '1285645143'
-- '1619231560'
-- '1962498394'
-- '1013296730'
-- '1417968157'
-- '1730419086'
-- '1386809150'
-- '1548325707'
-- '1609131770'
-- '1912106220'
-- '1912105826'
-- '1134153422'
-- '1790750404'
-- '1952425118'
-- '1952564015'
-- '1700815271'
-- '1942460290'
-- '1164622577'
-- '1801988654'
-- '1821404005'
-- '1740273374'
-- '1831295138'
-- '1003887803'
-- '1730167230'
-- '1154324739'
-- '1689765935'
-- '1598773939'
-- '1053726240'
-- '1144282377'
-- '1982660635'
-- '1790993228'
-- '1033129655'
-- '1316149636'
-- '1669451530'
-- '1568447514'
-- '1790999092'
-- '1720055163'
-- '1689766867'
-- '1811190408'
-- '1053497180'
-- '1306953468'
-- '1104260298'
-- '1790107217'
-- '1457591810'
-- '1558490912'
-- '1336336379'
-- '1831348952'
-- '1376509125'
-- '1316130735'
-- '1598098329'
-- '1750375663'
-- '1255862835'
-- '1588858138'
-- '1184816068'
-- '1366530107'
-- '1134206709'
-- '1598963902'
-- '1598723611'
-- '1073136057'
-- '1699213496'
-- '1508870874'
-- '1588850374'
-- '1285704635'
-- '1669678777'
-- '1124073309'
-- '1336229376'
-- '1780947655'
-- '1427497783'
-- '1902018591'
-- '1134184120'
-- '1588920748'
-- '1700901337'
-- '1750377651'
-- '1689992521'
-- '1962471045'
-- '1255599841'
-- '1265485692'
-- '1013206036'
-- '1669649927'
-- '1003062530'
-- '1851658470'
-- '1326289257'
-- '1497839120'
-- '1851370019'
-- '1124003975'
-- '1215092382'
-- '1326276296'
-- '1366703266'
-- '1528243359'
-- '1801984125'
-- '1225053093'
-- '1689644650'
-- '1407112170'
-- '1427340157'
-- '1699230664'
-- '1972874980'
-- '1386731776'
-- '1245482470'
-- '1154973675'
-- '1073670907'
-- '1184920423'
-- '1437391901'
-- '1326207986'
-- '1013013317'
-- '1932423225'
-- '1487006227'
-- '1982876058'
-- '1871626051'
-- '1609160027'
-- '1275357352'
-- '1639277478'
-- '1942265285'
-- '1013241470'
-- '1720271141'
-- '1255385985'
-- '1780898304'
-- '1396712774'
-- '1164454252'
-- '1891952545'
-- '1336489558'
-- '1326337288'
-- '1780856997'
-- '1790700797'
-- '1023541158'
-- '1538128210'
-- '1720178387'
-- '1215114962'
-- '1245281880'
-- '1699859652'
-- '1982794103'
-- '1598970436'
-- '1679747778'
-- '1912999848'
-- '1194941112'
-- '1790963924'
-- '1235319500'
-- '1164456216'
-- '1598292419'
-- '1396769840'
-- '1235155433'
-- '1336325810'
-- '1568543932'
-- '1851711121'
-- '1598785438'
-- '1851317176'
-- '1366642860'
-- '1063614659'
-- '1306007232'
-- '1740716125'
-- '1265621965'
-- '1427051168'
-- '1689646796'
-- '1073577938'
-- '1851310213'
-- '1871907121'
-- '1497996649'
-- '1295052827'
-- '1386689354'
-- '1811968134'
-- '1841204393'
-- '1275625683'
-- '1568482735'
-- '1053471482'
-- '1770803538'
-- '1740247436'
-- '1114455276'
-- '1134141039'
-- '1700140621'
-- '1790913036'
-- '1326171257'
-- '1114134301'
-- '1568588309'
-- '1730195983'
-- '1528273141'
-- '1609063957'
-- '1538397450'
-- '1023305018'
-- '1548367899'
-- '1063701274'
-- '1255646915'
-- '1477585123'
-- '1114022076'
-- '1306116389'
-- '1760448591'
-- '1821168279'
-- '1255351854'
-- '1417918046'
-- '1588819379'
-- '1760602643'
-- '1861433559'
-- '1457408338'
-- '1083928931'
-- '1336180355'
-- '1689692865'
-- '1144271925'
-- '1053738625'
-- '1265879266'
-- '1013175173'
-- '1932310224'
-- '1275643645'
-- '1386702587'
-- '1922022656'
-- '1437162138'
-- '1275746604'
-- '1861449159'
-- '1871561241'
-- '1851684542'
-- '1073752382'
-- '1326410481'
-- '1477529683'
-- '1154595346'
-- '1295826006'
-- '1588995179'
-- '1073728002'
-- '1558574491'
-- '1265460141'
-- '1861444366'
-- '1861689705'
-- '1366605438'
-- '1427449305'
-- '1669657748'
-- '1194097303'
-- '1346313590'
-- '1992052245'
-- '1376683300'
-- '1174547087'
-- '1558330332'
-- '1871529842'
-- '1881950301'
-- '1669573028'
-- '1689130601'
-- '1790047041'
-- '1972760056'
-- '1629041744'
-- '1639186075'
-- '1700061470'
-- '1023360914'
-- '1669669412'
-- '1235366337'
-- '1427475987'
-- '1528268950'
-- '1629033253'
-- '1548362361'
-- '1821288705'
-- '1417107285'
-- '1023097714'
-- '1851614473'
-- '1003882374'
-- '1326034422'
-- '1104881499'
-- '1497770598'
-- '1700189552'
-- '1922198605'
-- '1346268729'
-- '1093895476'
-- '1669752887'
-- '1942599154'
-- '1538189154'
-- '1417094772'
-- '1891024808'
-- '1356319867'
-- '1487880795'
-- '1619139284'
-- '1669408282'
-- '1629239744'
-- '1952620734'
-- '1447300470'
-- '1114292489'
-- '1851391577'
-- '1073946950'
-- '1972729499'
-- '1598050486'
-- '1023048972'
-- '1326275587'
-- '1922018548'
-- '1477840437'
-- '1710920566'
-- '1013994730'
-- '1710940705'
-- '1649389404'
-- '1780651893'
-- '1477644219'
-- '1821172560'
-- '1366098808'
-- '1689854838'
-- '1144266081'
-- '1306196266'
-- '1770920431'
-- '1124213178'
-- '1134156201'
-- '1467507244'
-- '1063463875'
-- '1548269822'
-- '1790005353'
-- '1740336346'
-- '1023060456'
-- '1083820518'
-- '1790719235'
-- '1235124322'
-- '1073678835'
-- '1134113244'
-- '1700871647'
-- '1457428500'
-- '1093138141'
-- '1568445104'
-- '1669459509'
-- '1750947115'
-- '1578934014'
-- '1386719185'
-- '1346534492'
-- '1235106246'
-- '1801942156'
-- '1912305434'
-- '1851369516'
-- '1770774341'
-- '1326369851'
-- '1285763227'
-- '1003802810'
-- '1407837586'
-- '1780725663'
-- '1083700314'
-- '1194868182'
-- '1043305204'
-- '1366799025'
-- '1447264932'
-- '1740418912'
-- '1477757086'
-- '1497060693'
-- '1932491404'
-- '1427377589'
-- '1649368168'
-- '1215956636'
-- '1467689547'
-- '1538463344'
-- '1104864263'
-- '1659565745'
-- '1669964029'
-- '1780774281'
-- '1659507259'
-- '1518027614'
-- '1225378334'
-- '1104032648'
-- '1053574988'
-- '1366478638'
-- '1487659314'
-- '1437204021'
-- '1043265077'
-- '1194013623'
-- '1871907642'
-- '1235559105'
-- '1487645693'
-- '1952412454'
-- '1205912540'
-- '1235367186'
-- '1609192699'
-- '1821409822'
-- '1871889295'
-- '1073685376'
-- '1679500565'
-- '1811925167'
-- '1053430850'
-- '1538249974'
-- '1356480966'
-- '1972693596'
-- '1942456165'
-- '1386977288'
-- '1639596679'
-- '1649200817'
-- '1962436964'
-- '1588863823'
-- '1790841641'
-- '1134474265'
-- '1710952015'
-- '1922063288'
-- '1659312759'
-- '1366049132'
-- '1639458961'
-- '1124117858'
-- '1083168066'
-- '1700810850'
-- '1992997977'
-- '1306072707'
-- '1952544272'
-- '1134193642'
-- '1275557191'
-- '1801292313'
-- '1891704771'
-- '1104916121'
-- '1841361607'
-- '1598847808'
-- '1659532117'
-- '1184670226'
-- '1619119187'
-- '1083737878'
-- '1124034053'
-- '1407114010'
-- '1013189729'
-- '1154519817'
-- '1750657748'
-- '1790809168'
-- '1013951540'
-- '1033168190'
-- '1437142478'
-- '1033553482'
-- '1487836995'
-- '1972988673'
-- '1295834000'
-- '1588666911'
-- '1861415739'
-- '1124203922'
-- '1215018023'
-- '1982699260'
-- '1124017926'
-- '1649474487'
-- '1023273141'
-- '1023144375'
-- '1265676274'
-- '1841332632'
-- '1497734677'
-- '1801969894'
-- '1790713352'
-- '1851357610'
-- '1518028430'
-- '1841255403'
-- '1477717932'
-- '1053561860'
-- '1518276872'
-- '1740297779'
-- '1821018623'
-- '1326113929'
-- '1811098965'
-- '1407210081'
-- '1629075635'
-- '1194046581'
-- '1275503047'
-- '1275743072'
-- '1295794923'
-- '1992767396'
-- '1649716424'
-- '1952428567'
-- '1407219298'
-- '1073923181'
-- '1336294107'
-- '1760438147'
-- '1003932963'
-- '1184957599'
-- '1508887852'
-- '1538508148'
-- '1689777476'
-- '1124213301'
-- '1619961117'
-- '1558528133'
-- '1104807239'
-- '1912172891'
-- '1750474029'
-- '1972857159'
-- '1699863811'
-- '1215146436'
-- '1083761407'
-- '1578552121'
-- '1922049022'
-- '1417248428'
-- '1487102661'
-- '1518071398'
-- '1881731909'
-- '1316985575'
-- '1487840443'
-- '1285687673'
-- '1235312885'
-- '1750588836'
-- '1972526457'
-- '1720437205'
-- '1962592998'
-- '1184675605'
-- '1215072855'
-- '1174739619'
-- '1033224944'
-- '1144336645'
-- '1386734317'
-- '1477670727'
-- '1568442242'
-- '1104980754'
-- '1407936818'
-- '1760632202'
-- '1528120680'
-- '1619134988'
-- '1669902185'
-- '1770663064'
-- '1952324998'
-- '1194794792'
-- '1114933272'
-- '1659815207'
-- '1811157308'
-- '1871850230'
-- '1972997807'
-- '1154321131'
-- '1477649531'
-- '1508132507'
-- '1114341310'
-- '1437400082'
-- '1609018084'
-- '1720200876'
-- '1225010044'
-- '1043272958'
-- '1467650911'
-- '1831258409'
-- '1164445177'
-- '1245320019'
-- '1457409377'
-- '1376934901'
-- '1386652030'
-- '1518055052'
-- '1871611962'
-- '1558550053'
-- '1417994781'
-- '1639224389'
-- '1659661692'
-- '1790214955'
-- '1508059098'
-- '1750744991'
-- '1154526523'
-- '1164678959'
-- '1417063389'
-- '1649347147'
-- '1114587342'
-- '1316039647'
-- '1326073131'
-- '1841413077'
-- '1124331426'
-- '1477681567'
-- '1043563588'
-- '1053361204'
-- '1891927216'
-- '1477651842'
-- '1740441815'
-- '1487745758'
-- '1609857010'
-- '1801846019'
-- '1952409138'
-- '1568481414'
-- '1134155344'
-- '1184917353'
-- '1871589119'
-- '1942275888'
-- '1386969715'
-- '1396061461'
-- '1972691756'
-- '1790977221'
-- '1487751947'
-- '1750301073'
-- '1275596405'
-- '1326068131'
-- '1336160456'
-- '1891823407'
-- '1013192426'
-- '1255324422'
-- '1649297359'
-- '1003254574'
-- '1841242989'
-- '1265407902'
-- '1508132457'
-- '1386907491'
-- '1831156066'
-- '1376852905'
-- '1376581702'
-- '1801897590'
-- '1598023509'
-- '1194959213'
-- '1255499646'
-- '1720166572'
-- '1861802415'
-- '1134544604'
-- '1447315791'
-- '1295791812'
-- '1760641088'
-- '1245253541'
-- '1366715351'
-- '1720304439'
-- '1053360776'
-- '1164413027'
-- '1205935632'
-- '1356399695'
-- '1003134677'
-- '1134534175'
-- '1427245018'
-- '1962527770'
-- '1356384374'
-- '1033512967'
-- '1184620205'
-- '1215154943'
-- '1598059735'
-- '1346367315'
-- '1801116496'
-- '1063690311'
-- '1891889754'
-- '1245349042'
-- '1922011162'
-- '1699743088'
-- '1457745341'
-- '1467848366'
-- '1447281878'
-- '1528585833'
-- '1215983382'
-- '1710145362'
-- '1790710531'
-- '1154431567'
-- '1386127579'
-- '1205896933'
-- '1114949617'
-- '1770949901'
-- '1104395656'
-- '1255538344'
-- '1568624633'
-- '1508914458'
-- '1992790778'
-- '1053539015'
-- '1134534597'
-- '1104010982'
-- '1437152832'
-- '1942545314'
-- '1831455690'
-- '1740218270'
-- '1407876584'
-- '1306926944'
-- '1366671380'
-- '1558796573'
-- '1912533555'
-- '1558523845'
-- '1477999522'
-- '1669821781'
-- '1306954268'
-- '1720528011'
-- '1467190454'
-- '1760561336'
-- '1851403539'
-- '1932519253'
-- '1700447216'
-- '1972554897'
-- '1811277130'
-- '1689651218'
-- '1679869143'
-- '1780635839'
-- '1346801164'
-- '1336260280'
-- '1225476930'
-- '1962481127'
-- '1275612749'
-- '1386893196'
-- '1720127194'
-- '1821103441'
-- '1861572117'
-- '1659459444'
-- '1104172584'
-- '1003203779'
-- '1407132665'
-- '1114294477'
-- '1902187859'
-- '1942612049'
-- '1881247260'
-- '1740400506'
-- '1609003011'
-- '1720407497'
-- '1861897241'
-- '1487958146'
-- '1326420662'
-- '1144230764'
-- '1497045298'
-- '1760473524'
-- '1255435301'
-- '1508401076'
-- '1629050851'
-- '1740296946'
-- '1134156599'
-- '1326085010'
-- '1538256979'
-- '1093232977'
-- '1861662975'
-- '1972765220'
-- '1740566561'
-- '1902219165'
-- '1386851335'
-- '1366406456'
-- '1013168673'
-- '1407878796'
-- '1609256064'
-- '1053890087'
-- '1255427324'
-- '1548266257'
-- '1194067470'
-- '1164988762'
-- '1225501885'
-- '1619905809'
-- '1891071478'
-- '1861888448'
-- '1114360997'
-- '1831476019'
-- '1912967365'
-- '1972868073'
-- '1154428027'
-- '1811934490'
-- '1609059005'
-- '1861667065'
-- '1073711966'
-- '1447207154'
-- '1508961897'
-- '1104906445'
-- '1710971999'
-- '1023492980'
-- '1134328982'
-- '1760552277'
-- '1801967997'
-- '1912511452'
-- '1386680718'
-- '1851382501'
-- '1497071963'
-- '1356429518'
-- '1811955479'
-- '1821479957'
-- '1043317910'
-- '1932429917'
-- '1164733960'
-- '1700015864'
-- '1245694652'
-- '1770900904'
-- '1346288461'
-- '1407860570'
-- '1033266820'
-- '1548420136'
-- '1699798603'
-- '1730239153'
-- '1144787359'
-- '1548350432'
-- '1316922875'
-- '1710900402'
-- '1942923289'
-- '1538522586'
-- '1043235591'
-- '1154641421'
-- '1861712358'
-- '1811160609'
-- '1790758837'
-- '1265451736'
-- '1417311788'
-- '1518996180'
-- '1013921253'
-- '1053396135'
-- '1346349487'
-- '1952367237'
-- '1306212741'
-- '1639691751'
-- '1255569174'
-- '1538111307'
-- '1396944674'
-- '1437151115'
-- '1164428165'
-- '1528225646'
-- '1063704112'
-- '1588735005'
-- '1245490325'
-- '1598960775'
-- '1477167146'
-- '1881733608'
-- '1982004305'
-- '1124057229'
-- '1649546185'
-- '1205292133'
-- '1205459252'
-- '1154561900'
-- '1639523830'
-- '1780782698'
-- '1811532278'
-- '1578505483'
-- '1790197929'
-- '1447295217'
-- '1073709457'
-- '1013470186'
-- '1548644883'
-- '1669720017'
-- '1831396936'
-- '1558725952'
-- '1467185751'
-- '1154594885'
-- '1124405311'
-- '1457330607'
-- '1184001596'
-- '1578837134'
-- '1144866443'
-- '1760413991'
-- '1841685419'
-- '1265694574'
-- '1093240343'
-- '1619970787'
-- '1740202159'
-- '1164538690'
-- '1083221717'
-- '1992779441'
-- '1013086735'
-- '1144335761'
-- '1790216083'
-- '1518315779'
-- '1164691549'
-- '1699700740'
-- '1760652267'
-- '1457797060'
-- '1427205269'
-- '1831276765'
-- '1194776641'
-- '1710388830'
-- '1699983155'
-- '1174965438'
-- '1063647071'
-- '1215130612'
-- '1992541007'
-- '1669543922'
-- '1912266339'
-- '1114331188'
-- '1063904027'
-- '1629528435'
-- '1417440595'
-- '1174849046'
-- '1699088104'
-- '1053840074'
-- '1326209461'
-- '1548618663'
-- '1790812642'
-- '1699074484'
-- '1548559974'
-- '1093155632'
-- '1255743886'
-- '1013945708'
-- '1750397261'
-- '1710520671'
-- '1831286483'
-- '1942568449'
-- '1558558254'
-- '1912970856'
-- '1962550855'
-- '1427154673'
-- '1477531028'
-- '1003050980'
-- '1538409370'
-- '1457699118'
-- '1922098664'
-- '1861445512'
-- '1811945173'
-- '1891261699'
-- '1639279474'
-- '1275099954'
-- '1023121159'
-- '1770887788'
-- '1851741318'
-- '1205049608'
-- '1568536183'
-- '1366428625'
-- '1235190968'
-- '1780931956'
-- '1083782312'
-- '1669850780'
-- '1073726782'
-- '1568761427'
-- '1962020701'
-- '1285710269'
-- '1174814560'
-- '1477915676'
-- '1851566210'
-- '1134199946'
-- '1427290741'
-- '1609304211'
-- '1831730357'
-- '1770877904'
-- '1578520250'
-- '1801812128'
-- '1568548246'
-- '1366433211'
-- '1154521722'
-- '1750905022'
-- '1831348176'
-- '1750413050'
-- '1538335583'
-- '1609800804'
-- '1437443033'
-- '1275728313'
-- '1245766757'
-- '1740387273'
-- '1255857181'
-- '1609941806'
-- '1003287186'
-- '1285306324'
-- '1982674123'
-- '1245484583'
-- '1184093320'
-- '1376521278'
-- '1982649133'
-- '1003828179'
-- '1891759312'
-- '1184858813'
-- '1568728806'
-- '1053364000'
-- '1962813857'
-- '1376098095'
-- '1003800640'
-- '1700872710'
-- '1902324528'
-- '1225113954'
-- '1194986554'
-- '1013293380'
-- '1033154083'
-- '1083051122'
-- '1780045690'
-- '1104010776'
-- '1851875561'
-- '1023200821'
-- '1821266966'
-- '1588984561'
-- '1386686467'
-- '1376651596'
-- '1194043307'
-- '1386900694'
-- '1801811351'
-- '1316250491'
-- '1801883871'
-- '1093078305'
-- '1871599050'
-- '1831135276'
-- '1134112436'
-- '1073574307'
-- '1831338755'
-- '1073661997'
-- '1093945800'
-- '1053708065'
-- '1487934204'
-- '1154371946'
-- '1013116615'
-- '1073687059'
-- '1437211299'
-- '1477086668'
-- '1376889998'
-- '1154883148'
-- '1487698874'
-- '1376932475'
-- '1083782932'
-- '1851612584'
-- '1902012180'
-- '1134149495'
-- '1902387210'
-- '1568469971'
-- '1487784864'
-- '1831429752'
-- '1083117766'
-- '1457439713'
-- '1356353791'
-- '1376035535'
-- '1548623135'
-- '1760877088'
-- '1932307618'
-- '1447411780'
-- '1427438811'
-- '1386671394'
-- '1457348872'
-- '1033286091'
-- '1639493141'
-- '1982869103'
-- '1124556808'
-- '1801086269'
-- '1528188182'
-- '1245657402'
-- '1336553288'
-- '1053491472'
-- '1801047352'
-- '1710009642'
-- '1093979700'
-- '1427546670'
-- '1366602344'
-- '1386631455'
-- '1245692557'
-- '1386703999'
-- '1356411474'
-- '1801474887'
-- '1396239992'
-- '1215370689'
-- '1689090540'
-- '1275595431'
-- '1962709477'
-- '1003991746'
-- '1821627365'
-- '1003985649'
-- '1992854392'
-- '1043327620'
-- '1073154209'
-- '1700847860'
-- '1407210891'
-- '1790144384'
-- '1760916860'
-- '1730141003'
-- '1184646119'
-- '1790076057'
-- '1790912665'
-- '1063586485'
-- '1598804528'
-- '1548656911'
-- '1215223409'
-- '1982650453'
-- '1720222078'
-- '1245486141'
-- '1922005677'
-- '1902004674'
-- '1821032129'
-- '1336248996'
-- '1871868984'
-- '1841258696'
-- '1437389426'
-- '1962528034'
-- '1831164417'
-- '1548299548'
-- '1083841191'
-- '1972706315'
-- '1790747715'
-- '1013242106'
-- '1902864465'
-- '1457381519'
-- '1356394324'
-- '1629250394'
-- '1255328670'
-- '1912087354'
-- '1972786895'
-- '1275500092'
-- '1780736702'
-- '1720084114'
-- '1285969162'
-- '1891967964'
-- '1942292230'
-- '1225133812'
-- '1831416452'
-- '1063663573'
-- '1093773442'
-- '1063405165'
-- '1215118625'
-- '1265489652'
-- '1295709913'
-- '1275821555'
-- '1497958243'
-- '1154515963'
-- '1093157455'
-- '1619240629'
-- '1033202825'
-- '1619233228'
-- '1649560012'
-- '1356372783'
-- '1407896590'
-- '1225238355'
-- '1831240100'
-- '1205951126'
-- '1043538150'
-- '1386218501'
-- '1598186413'
-- '1902906852'
-- '1962897348'
-- '1184005357'
-- '1154586659'
-- '1730295031'
-- '1649775263'
-- '1841679974'
-- '1699069591'
-- '1861589996'
-- '1295946176'
-- '1972714491'
-- '1689758831'
-- '1912071937'
-- '1194791814'
-- '1477530319'
-- '1750461802'
-- '1245824176'
-- '1093784530'
-- '1619979697'
-- '1558548644'
-- '1841233988'
-- '1386637114'
-- '1154387041'
-- '1699922369'
-- '1841867512'
-- '1750693719'
-- '1003026139'
-- '1376836783'
-- '1598024754'
-- '1285806943'
-- '1568858264'
-- '1174660633'
-- '1750317806'
-- '1669676771'
-- '1538697537'
-- '1760440127'
-- '1578182465'
-- '1093292195'
-- '1679686653'
-- '1578022968'
-- '1295713782'
-- '1649247339'
-- '1336526920'
-- '1215270087'
-- '1942542253'
-- '1285896647'
-- '1851988117'
-- '1073199089'
-- '1679791370'
-- '1164470126'
-- '1992801294'
-- '1093754129'
-- '1851335723'
-- '1366829152'
-- '1992148290'
-- '1639179229'
-- '1558858092'
-- '1326082686'
-- '1316033871'
-- '1508363151'
-- '1801316526'
-- '1497062566'
-- '1710419346'
-- '1326400870'
-- '1013455104'
-- '1508828880'
-- '1821256371'
-- '1821309410'
-- '1558442715'
-- '1184952038'
-- '1255864724'
-- '1225371636'
-- '1740274281'
-- '1396709218'
-- '1609262559'
-- '1760601090'
-- '1407172521'
-- '1942246533'
-- '1619728318'
-- '1568411346'
-- '1184709941'
-- '1023456357'
-- '1891413464'
-- '1194925933'
-- '1316235716'
-- '1831109792'
-- '1669959441'
-- '1720641665'
-- '1043662299'
-- '1043375363'
-- '1356380877'
-- '1902532047'
-- '1447348388'
-- '1164494050'
-- '1609313295'
-- '1245972637'
-- '1639586472'
-- '1720498959'
-- '1922386119'
-- '1073544722'
-- '1508941246'
-- '1720798556'
-- '1841451010'
-- '1285315531'
-- '1265276315'
-- '1396829172'
-- '1205899259'
-- '1134461262'
-- '1720237290'
-- '1871040816'
-- '1992122931'
-- '1467686683'
-- '1801956099'
-- '1356474548'
-- '1295778843'
-- '1629501481'
-- '1831466705'
-- '1891052171'
-- '1033459599'
-- '1912616442'
-- '1770800807'
-- '1447690938'
-- '1629509153'
-- '1124373683'
-- '1013999291'
-- '1902006257'
-- '1588827026'
-- '1104386291'
-- '1386890978'
-- '1508194341'
-- '1811162217'
-- '1508866542'
-- '1417943002'
-- '1144922410'
-- '1063674133'
-- '1588984652'
-- '1609578392'
-- '1104056597'
-- '1508268723'
-- '1982138483'
-- '1215923115'
-- '1154999787'
-- '1629572078'
-- '1861463390'
-- '1578805818'
-- '1508820069'
-- '1508125295'
-- '1154685386'
-- '1336821818'
-- '1033538996'
-- '1528565884'
-- '1376854521'
-- '1861629172'
-- '1396938106'
-- '1578523320'
-- '1124417142'
-- '1033106091'
-- '1093877102'
-- '1063694552'
-- '1134455082'
-- '1386719847'
-- '1336680404'
-- '1245291707'
-- '1578538013'
-- '1285696203'
-- '1770950438'
-- '1124002209'
-- '1891933768'
-- '1962441998'
-- '1598792640'
-- '1245349018'
-- '1487880738'
-- '1285260216'
-- '1144498429'
-- '1659662591'
-- '1508364043'
-- '1538193669'
-- '1821300138'
-- '1538118542'
-- '1629086145'
-- '1598847840'
-- '1548362759'
-- '1891774071'
-- '1760913057'
-- '1669910402'
-- '1134124142'
-- '1265400337'
-- '1568410793'
-- '1124011150'
-- '1508023029'
-- '1245220672'
-- '1275841603'
-- '1528046810'
-- '1396826533'
-- '1114963808'
-- '1851348700'
-- '1235358623'
-- '1457509523'
-- '1568660728'
-- '1356313258'
-- '1639124530'
-- '1831138155'
-- '1649405549'
-- '1376864470'
-- '1871586925'
-- '1861471070'
-- '1427266055'
-- '1568423879'
-- '1376596296'
-- '1922077841'
-- '1902554744'
-- '1922065275'
-- '1710070941'
-- '1548226061'
-- '1952763336'
-- '1942625421'
-- '1184882862'
-- '1184692360'
-- '1912970104'
-- '1306826870'
-- '1396060166'
-- '1114217981'
-- '1104863976'
-- '1568805752'
-- '1255561023'
-- '1760709497'
-- '1134668478'
-- '1326012766'
-- '1689949851'
-- '1851610299'
-- '1760435184'
-- '1629231451'
-- '1356840441'
-- '1891794525'
-- '1225203672'
-- '1932162401'
-- '1265478481'
-- '1003922139'
-- '1831164672'
-- '1134211378'
-- '1235409673'
-- '1043271943'
-- '1144563602'
-- '1811075567'
-- '1851493373'
-- '1366935439'
-- '1801174453'
-- '1013327394'
-- '1174847040'
-- '1801229505'
-- '1851788822'
-- '1750486890'
-- '1194717942'
-- '1013283043'
-- '1033139738'
-- '1477573467'
-- '1134169063'
-- '1902079015'
-- '1518912047'
-- '1629428123'
-- '1841346111'
-- '1811911191'
-- '1356798276'
-- '1235118308'
-- '1497742902'
-- '1609866938'
-- '1003889049'
-- '1871547042'
-- '1457677619'
-- '1508156795'
-- '1801815691'
-- '1942565916'
-- '1477808004'
-- '1235188202'
-- '1538135454'
-- '1285096339'
-- '1295756336'
-- '1790774511'
-- '1669771028'
-- '1154514784'
-- '1821244286'
-- '1629380522'
-- '1528223682'
-- '1770519886'
-- '1578523437'
-- '1154379550'
-- '1083976138'
-- '1578703633'
-- '1093970493'
-- '1760451462'
-- '1497046783'
-- '1235133737'
-- '1992718639'
-- '1114913704'
-- '1821286857'
-- '1982773461'
-- '1295113835'
-- '1568653558'
-- '1467529222'
-- '1013176197'
-- '1750575346'
-- '1952424079'
-- '1023270196'
-- '1679968168'
-- '1013999531'
-- '1750543336'
-- '1063702165'
-- '1639602964'
-- '1407804834'
-- '1841672961'
-- '1407819360'
-- '1033487160'
-- '1730440413'
-- '1760831937'
-- '1588629315'
-- '1245222496'
-- '1578728085'
-- '1023120136'
-- '1275809923'
-- '1538629571'
-- '1780664722'
-- '1518157247'
-- '1487761631'
-- '1700128725'
-- '1740570803'
-- '1205804424'
-- '1356370415'
-- '1528328713'
-- '1821645706'
-- '1972553485'
-- '1104138577'
-- '1427496850'
-- '1841590957'
-- '1912993635'
-- '1619952421'
-- '1710127873'
-- '1972700011'
-- '1295729697'
-- '1215937552'
-- '1366843807'
-- '1639135015'
-- '1831398601'
-- '1568669703'
-- '1427554765'
-- '1356421176'
-- '1023195096'
-- '1174510341'
-- '1518931492'
-- '1609282607'
-- '1275516361'
-- '1477797603'
-- '1467497859'
-- '1497930176'
-- '1942444336'
-- '1023531043'
-- '1558536698'
-- '1891458691'
-- '1538111273'
-- '1104853274'
-- '1265473292'
-- '1033539671'
-- '1144231671'
-- '1184622359'
-- '1437201100'
-- '1467598508'
-- '1104081645'
-- '1114944139'
-- '1881813194'
-- '1619956547'
-- '1043426349'
-- '1144847005'
-- '1912034364'
-- '1023544343'
-- '1265792295'
-- '1962437608'
-- '1215196050'
-- '1467890632'
-- '1922462019'
-- '1912962168'
-- '1346528742'
-- '1164419107'
-- '1164689477'
-- '1386641678'
-- '1447447164'
-- '1811985989'
-- '1912975632'
-- '1275709958'
-- '1427143825'
-- '1467544312'
-- '1548280340'
-- '1861407348'
-- '1154360147'
-- '1457315368'
-- '1710561782'
-- '1871754135'
-- '1134189038'
-- '1275628752'
-- '1366509226'
-- '1528097854'
-- '1093036394'
-- '1427411131'
-- '1902844418'
-- '1154720027'
-- '1891434817'
-- '1437237971'
-- '1114112026'
-- '1689870982'
-- '1992766695'
-- '1639205354'
-- '1275514135'
-- '1144715798'
-- '1750353777'
-- '1982997417'
-- '1609928837'
-- '1700047701'
-- '1003969429'
-- '1194887125'
-- '1962607820'
-- '1548317431'
-- '1770794182'
-- '1447417712'
-- '1043455157'
-- '1871607754'
-- '1902199508'
-- '1114914397'
-- '1174605414'
-- '1487811907'
-- '1740324292'
-- '1326207788'
-- '1700800448'
-- '1790959294'
-- '1013262740'
-- '1881245504'
-- '1851366801'
-- '1538943238'
-- '1952501439'
-- '1932191053'
-- '1821482787'
-- '1821387549'
-- '1952546384'
-- '1215943923'
-- '1114929650'
-- '1417218926'
-- '1144282559'
-- '1407882368'
-- '1457476970'
-- '1124551262'
-- '1194343459'
-- '1073511424'
-- '1235622176'
-- '1700887312'
-- '1346736147'
-- '1548854219'
-- '1144391889'
-- '1255548327'
-- '1881973543'
-- '1023285368'
-- '1558458067'
-- '1609971290'
-- '1427203140'
-- '1265782817'
-- '1114965241'
-- '1366619645'
-- '1639466956'
-- '1225282882'
-- '1265401905'
-- '1346549847'
-- '1245455450'
-- '1568489151'
-- '1982618872'
-- '1063562924'
-- '1629428552'
-- '1366537177'
-- '1104204767'
-- '1760644207'
-- '1003014788'
-- '1114967932'
-- '1841432457'
-- '1306800404'
-- '1053426494'
-- '1164829768'
-- '1689708208'
-- '1164609970'
-- '1316962640'
-- '1770533291'
-- '1457585010'
-- '1154460947'
-- '1588868459'
-- '1952744898'
-- '1891715405'
-- '1174675318'
-- '1316388655'
-- '1679584106'
-- '1780156778'
-- '1225298086'
-- '1801824487'
-- '1922354166'
-- '1386800027'
-- '1568459642'
-- '1255783411'
-- '1003072760'
-- '1144544669'
-- '1134305469'
-- '1346267937'
-- '1588810568'
-- '1942628839'
-- '1689644528'
-- '1407829286'
-- '1437396546'
-- '1366414047'
-- '1427290055'
-- '1386868982'
-- '1417400821'
-- '1447228267'
-- '1164531927'
-- '1033139480'
-- '1376633826'
-- '1912062324'
-- '1578839650'
-- '1033176912'
-- '1699736892'
-- '1790163434'
-- '1871866913'
-- '1306821707'
-- '1326129289'
-- '1326135476'
-- '1447235684'
-- '1861854630'
-- '1194985002'
-- '1447362819'
-- '1538394275'
-- '1922164870'
-- '1144339763'
-- '1104832823'
-- '1134189970'
-- '1710050778'
-- '1104180579'
-- '1982031571'
-- '1366575037'
-- '1407191273'
-- '1760574552'
-- '1841380870'
-- '1447454921'
-- '1306163092'
-- '1083682314'
-- '1528017258'
-- '1538255468'
-- '1417205329'
-- '1790857118'
-- '1992848733'
-- '1669415535'
-- '1720518921'
-- '1013941558'
-- '1700833753'
-- '1710197181'
-- '1265479745'
-- '1952310765'
-- '1104923374'
-- '1356407068'
-- '1821206269'
-- '1861713083'
-- '1205892445'
-- '1467766311'
-- '1518604933'
-- '1164997623'
-- '1588123442'
-- '1417230996'
-- '1780609131'
-- '1902048325'
-- '1013901768'
-- '1407984362'
-- '1831532894'
-- '1871617621'
-- '1942380217'
-- '1548250863'
-- '1245576487'
-- '1235577230'
-- '1619067022'
-- '1083146567'
-- '1730434234'
-- '1114027190'
-- '1427316231'
-- '1538322755'
-- '1932189255'
-- '1912072992'
-- '1053408708'
-- '1003006248'
-- '1760477582'
-- '1558447631'
-- '1932416740'
-- '1932304177'
-- '1144261868'
-- '1174574586'
-- '1386639425'
-- '1285841387'
-- '1477519692'
-- '1679617930'
-- '1780029504'
-- '1033352174'
-- '1336229673'
-- '1194956482'
-- '1578669743'
-- '1407899529'
-- '1588914857'
-- '1225078397'
-- '1821070459'
-- '1972590354'
-- '1265792386'
-- '1780652354'
-- '1194833376'
-- '1437513173'
-- '1861466674'
-- '1902309255'
-- '1063683209'
-- '1194951756'
-- '1457442725'
-- '1104271436'
-- '1194973479'
-- '1891106142'
-- '1700044336'
-- '1962437483'
-- '1144651183'
-- '1811108426'
-- '1992806210'
-- '1750326377'
-- '1891991295'
-- '1346248465'
-- '1124098538'
-- '1528572138'
-- '1073570586'
-- '1669465357'
-- '1841239779'
-- '1225396716'
-- '1194068940'
-- '1720001746'
-- '1952348120'
-- '1346451655'
-- '1013352657'
-- '1609019470'
-- '1528387610'
-- '1477799633'
-- '1568663623'
-- '1326122441'
-- '1437185972'
-- '1285950527'
-- '1467530709'
-- '1386092955'
-- '1265588925'
-- '1265779029'
-- '1912918111'
-- '1104097765'
-- '1245585017'
-- '1427477744'
-- '1750361820'
-- '1477765113'
-- '1053376632'
-- '1922156801'
-- '1366467649'
-- '1598701898'
-- '1750393617'
-- '1598758633'
-- '1174928162'
-- '1033592795'
-- '1295776292'
-- '1639388994'
-- '1346231859'
-- '1952573388'
-- '1770835548'
-- '1003998816'
-- '1225221641'
-- '1437369840'
-- '1821260845'
-- '1245571553'
-- '1467488387'
-- '1609010420'
-- '1609294016'
-- '1003923038'
-- '1336198126'
-- '1356768816'
-- '1023668605'
-- '1134191844'
-- '1093726655'
-- '1902016967'
-- '1780657122'
-- '1578083002'
-- '1598783300'
-- '1588262133'
-- '1013912674'
-- '1063602928'
-- '1801131347'
-- '1932393915'
-- '1144789009'
-- '1851300305'
-- '1811062813'
-- '1972034841'
-- '1821037912'
-- '1295930527'
-- '1619058237'
-- '1629169107'
-- '1790716595'
-- '1053637694'
-- '1699075333'
-- '1467441295'
-- '1639308182'
-- '1578792495'
-- '1679967640'
-- '1992868772'
-- '1245650878'
-- '1861423246'
-- '1962921528'
-- '1366008104'
-- '1689734741'
-- '1053308817'
-- '1275855595'
-- '1952362246'
-- '1346689916'
-- '1043487101'
-- '1700166881'
-- '1982799417'
-- '1609185990'
-- '1710518964'
-- '1508916321'
-- '1780983460'
-- '1740491034'
-- '1154344760'
-- '1215996350'
-- '1639362114'
-- '1649396235'
-- '1295757755'
-- '1942216338'
-- '1811099633'
-- '1881659696'
-- '1881852051'
-- '1528021755'
-- '1306076039'
-- '1376819474'
-- '1427085810'
-- '1427475417'
-- '1043317662'
-- '1184685273'
-- '1275871964'
-- '1649246109'
-- '1386852028'
-- '1548618705'
-- '1275989550'
-- '1356357941'
-- '1457779357'
-- '1740423771'
-- '1558476747'
-- '1083622518'
-- '1295052611'
-- '1790277523'
-- '1205888906'
-- '1730277658'
-- '1902099658'
-- '1326354853'
-- '1881988517'
-- '1306102546'
-- '1578345351'
-- '1104873199'
-- '1164761680'
-- '1558788042'
-- '1003252800'
-- '1164660304'
-- '1811911530'
-- '1053543165'
-- '1245269778'
-- '1861501116'
-- '1871794172'
-- '1164591657'
-- '1922215805'
-- '1407847312'
-- '1922114297'
-- '1093052029'
-- '1639311905'
-- '1760468847'
-- '1235187790'
-- '1437234531'
-- '1669772935'
-- '1700145976'
-- '1972645984'
-- '1245557891'
-- '1891701066'
-- '1538362371'
-- '1265447767'
-- '1295050060'
-- '1689835209'
-- '1083902563'
-- '1326284548'
-- '1750369468'
-- '1245827401'
-- '1427270313'
-- '1699797845'
-- '1710275722'
-- '1659481679'
-- '1942592738'
-- '1972800837'
-- '1902826498'
-- '1528142346'
-- '1083013254'
-- '1376727768'
-- '1548491699'
-- '1770549651'
-- '1114510112'
-- '1437398955'
-- '1437323573'
-- '1457391468'
-- '1881824795'
-- '1891055919'
-- '1568427516'
-- '1700197795'
-- '1801013842'
-- '1588815039'
-- '1073785838'
-- '1366632408'
-- '1801946447'
-- '1841395167'
-- '1982704276'
-- '1669697546'
-- '1811208655'
-- '1407075435'
-- '1679588313'
-- '1497778237'
-- '1366552812'
-- '1134237100'
-- '1245434612'
-- '1275277824'
-- '1568482719'
-- '1629005368'
-- '1255350583'
-- '1699841098'
-- '1417063777'
-- '1548471907'
-- '1306860036'
-- '1538400593'
-- '1194815191'
-- '1689659955'
-- '1851616908'
-- '1073709648'
-- '1467712794'
-- '1811542723'
-- '1912967894'
-- '1013337096'
-- '1255657664'
-- '1124248067'
-- '1891258331'
-- '1972605806'
-- '1851406284'
-- '1871687087'
-- '1578513701'
-- '1114223773'
-- '1467984112'
-- '1477573640'
-- '1295964682'
-- '1396706479'
-- '1417188061'
-- '1942304357'
-- '1407883184'
-- '1083894315'
-- '1629229836'
-- '1043449457'
-- '1467427229'
-- '1851557813'
-- '1528228491'
-- '1487297651'
-- '1346852597'
-- '1699799684'
-- '1033176235'
-- '1457358962'
-- '1558749796'
-- '1083199228'
-- '1639442262'
-- '1467498170'
-- '1043659550'
-- '1639171242'
-- '1194281949'
-- '1588661177'
-- '1235121963'
-- '1083027676'
-- '1790215994'
-- '1447638689'
-- '1912001637'
-- '1720375520'
-- '1306847397'
-- '1053872879'
-- '1427493071'
-- '1780670158'
-- '1497717144'
-- '1912492323'
-- '1821243379'
-- '1225092422'
-- '1235704321'
-- '1821265646'
-- '1003831736'
-- '1760446918'
-- '1366836975'
-- '1811951486'
-- '1316333461'
-- '1104188952'
-- '1306884879'
-- '1417377946'
-- '1922445568'
-- '1245234822'
-- '1902932767'
-- '1871593707'
-- '1528259991'
-- '1629394879'
-- '1720405103'
-- '1245255611'
-- '1639388424'
-- '1437655990'
-- '1720372683'
-- '1144286550'
-- '1437133634'
-- '1306864962'
-- '1629080833'
-- '1558723254'
-- '1013173376'
-- '1932542909'
-- '1912341231'
-- '1497974729'
-- '1497071443'
-- '1457322125'
-- '1710321617'
-- '1003130881'
-- '1548452493'
-- '1760447494'
-- '1003870254'
-- '1023077708'
-- '1457664120'
-- '1740237759'
-- '1346206018'
-- '1770515819'
-- '1689676934'
-- '1609260785'
-- '1154551463'
-- '1053408716'
-- '1376105122'
-- '1013932029'
-- '1588824155'
-- '1023250727'
-- '1093790925'
-- '1215297049'
-- '1235389784'
-- '1497773998'
-- '1730220617'
-- '1821031295'
-- '1831693167'
-- '1740624766'
-- '1154663482'
-- '1831509421'
-- '1902105778'
-- '1922106301'
-- '1457384398'
-- '1902899248'
-- '1861729709'
-- '1710978143'
-- '1679993034'
-- '1427478684'
-- '1922323856'
-- '1770579872'
-- '1245689447'
-- '1609140490'
-- '1730147869'
-- '1629273412'
-- '1578750485'
-- '1134260821'
-- '1275537847'
-- '1316047830'
-- '1013016781'
-- '1760610018'
-- '1902849847'
-- '1609040781'
-- '1255596508'
-- '1346603461'
-- '1285083220'
-- '1548554678'
-- '1326481763'
-- '1073570495'
-- '1114064615'
-- '1548290976'
-- '1316008865'
-- '1558381707'
-- '1740290964'
-- '1104186584'
-- '1619088945'
-- '1265965610'
-- '1962665984'
-- '1497861058'
-- '1558672287'
-- '1952968240'
-- '1396235750'
-- '1851417794'
-- '1073760930'
-- '1861420507'
-- '1285714527'
-- '1538222401'
-- '1629614466'
-- '1912993650'
-- '1821208026'
-- '1053738401'
-- '1932146115'
-- '1730177916'
-- '1174572689'
-- '1073637617'
-- '1275824500'
-- '1487880621'
-- '1932119864'
-- '1740222371'
-- '1902982275'
-- '1326364795'
-- '1336202241'
-- '1568433506'
-- '1215207238'
-- '1578556437'
-- '1043271851'
-- '1144303439'
-- '1316104730'
-- '1992332514'
-- '1063654952'
-- '1518264647'
-- '1013411149'
-- '1114421070'
-- '1427402031'
-- '1932529633'
-- '1366418238'
-- '1851445506'
-- '1407180938'
-- '1770788143'
-- '1679683080'
-- '1942283759'
-- '1104314236'
-- '1043673528'
-- '1356738850'
-- '1497791032'
-- '1174512990'
-- '1649699240'
-- '1700927548'
-- '1700957008'
-- '1528467552'
-- '1144273137'
-- '1336300953'
-- '1447604442'
-- '1710086400'
-- '1902164916'
-- '1902912397'
-- '1730243965'
-- '1528266053'
-- '1457672982'
-- '1245431915'
-- '1225006091'
-- '1861476061'
-- '1154944619'
-- '1841556305'
-- '1477568996'
-- '1992141741'
-- '1639339419'
-- '1417309360'
-- '1699904607'
-- '1972505097'
-- '1528314259'
-- '1053383000'
-- '1609945997'
-- '1538166681'
-- '1164626032'
-- '1366703340'
-- '1043653454'
-- '1871175737'
-- '1255673455'
-- '1114189552'
-- '1285653956'
-- '1699705541'
-- '1831104371'
-- '1487876306'
-- '1417989203'
-- '1053548446'
-- '1063826501'
-- '1023437936'
-- '1952488686'
-- '1558628552'
-- '1407155864'
-- '1760839849'
-- '1760626410'
-- '1528155082'
-- '1841290780'
-- '1881655520'
-- '1245200534'
-- '1396274908'
-- '1821625419'
-- '1003912684'
-- '1619170396'
-- '1013988682'
-- '1205906518'
-- '1427288588'
-- '1619134178'
-- '1790876480'
-- '1245259324'
-- '1770844516'
-- '1780117549'
-- '1053307215'
-- '1447639224'
-- '1780644369'
-- '1154385250'
-- '1730437930'
-- '1760619142'
-- '1164712238'
-- '1023122553'
-- '1164672788'
-- '1437501046'
-- '1457801359'
-- '1952602278'
-- '1003294844'
-- '1467486712'
-- '1669892055'
-- '1952695017'
-- '1194012179'
-- '1629205752'
-- '1760490239'
-- '1811077084'
-- '1952534364'
-- '1972579233'
-- '1154525244'
-- '1245379270'
-- '1912990201'
-- '1104298090'
-- '1326208844'
-- '1265444749'
-- '1255311692'
-- '1306804422'
-- '1790179208'
-- '1700016482'
-- '1861453391'
-- '1407031354'
-- '1598748873'
-- '1841601895'
-- '1093823965'
-- '1518274299'
-- '1619935285'
-- '1679898357'
-- '1396195103'
-- '1700972536'
-- '1023620341'
-- '1073564373'
-- '1245572684'
-- '1477117984'
-- '1518487156'
-- '1003890807'
-- '1487928735'
-- '1114457116'
-- '1154434868'
-- '1669715322'
-- '1134488471'
-- '1295291938'
-- '1568052918'
-- '1013263797'
-- '1235495698'
-- '1184686990'
-- '1720212244'
-- '1023338381'
-- '1104839497'
-- '1134337249'
-- '1194145870'
-- '1417342080'
-- '1467488528'
-- '1770731374'
-- '1578737532'
-- '1619647138'
-- '1073789905'
-- '1114118452'
-- '1366407488'
-- '1730236548'
-- '1619340064'
-- '1790065738'
-- '1508398785'
-- '1093715831'
-- '1215950282'
-- '1811255284'
-- '1053660688'
-- '1528359650'
-- '1639794100'
-- '1760645428'
-- '1770749749'
-- '1851580674'
-- '1679576706'
-- '1730498262'
-- '1114697885'
-- '1174882385'
-- '1396126017'
-- '1972565851'
-- '1700318128'
-- '1912206657'
-- '1417942590'
-- '1083749998'
-- '1194987461'
-- '1053574160'
-- '1073856696'
-- '1598992224'
-- '1689646366'
-- '1225324387'
-- '1447348883'
-- '1841556354'
-- '1851441893'
-- '1992827380'
-- '1497078802'
-- '1245471275'
-- '1750510244'
-- '1831491836'
-- '1073733861'
-- '1770016735'
-- '1821309683'
-- '1093131302'
-- '1316115777'
-- '1720130628'
-- '1225291974'
-- '1700910114'
-- '1770110108'
-- '1851465272'
-- '1073859070'
-- '1255411500'
-- '1295796373'
-- '1366415986'
-- '1568480093'
-- '1730247974'
-- '1891813978'
-- '1437243102'
-- '1487069910'
-- '1992360697'
-- '1194017855'
-- '1720597081'
-- '1780883405'
-- '1104053388'
-- '1578653853'
-- '1043505415'
-- '1730758608'
-- '1013540327'
-- '1063435774'
-- '1407465610'
-- '1790876043'
-- '1891993572'
-- '1164633723'
-- '1093923450'
-- '1609882414'
-- '1053403832'
-- '1326081514'
-- '1386023851'
-- '1487684809'
-- '1972697233'
-- '1053334466'
-- '1154940849'
-- '1871684142'
-- '1972712743'
-- '1730113820'
-- '1730261611'
-- '1003880329'
-- '1407099138'
-- '1467959072'
-- '1811158272'
-- '1952729329'
-- '1720106800'
-- '1801198957'
-- '1003319054'
-- '1629297924'
-- '1760440168'
-- '1811997257'
-- '1417216706'
-- '1447226832'
-- '1588754089'
-- '1477566149'
-- '1568966893'
-- '1629136692'
-- '1770741555'
-- '1205007994'
-- '1285081463'
-- '1609805910'
-- '1023276763'
-- '1053756452'
-- '1245469808'
-- '1982810669'
-- '1144466558'
-- '1205863669'
-- '1972813004'
-- '1972957652'
-- '1962611277'
-- '1396436614'
-- '1447696703'
-- '1528148681'
-- '1700852977'
-- '1174156046'
-- '1053706135'
-- '1366684490'
-- '1437174166'
-- '1093112187'
-- '1790008993'
-- '1386966984'
-- '1528481181'
-- '1881693687'
-- '1184633968'
-- '1548383763'
-- '1063797017'
-- '1285393835'
-- '1518291145'
-- '1679864201'
-- '1811496177'
-- '1083605752'
-- '1588711394'
-- '1811152796'
-- '1851705099'
-- '1912286493'
-- '1144985383'
-- '1295901858'
-- '1376747550'
-- '1851655310'
-- '1144278136'
-- '1619373537'
-- '1295768109'
-- '1982794509'
-- '1588865414'
-- '1992122212'
-- '1548338221'
-- '1659581361'
-- '1821302043'
-- '1962662726'
-- '1093243941'
-- '1477758050'
-- '1902986334'
-- '1942463682'
-- '1124098819'
-- '1194233619'
-- '1245344084'
-- '1801240908'
-- '1548486715'
-- '1265449532'
-- '1508457730'
-- '1689845687'
-- '1437244910'
-- '1821287301'
-- '1174880496'
-- '1396718078'
-- '1225328396'
-- '1417021106'
-- '1679839914'
-- '1326246521'
-- '1083634828'
-- '1255532669'
-- '1578701710'
-- '1003889692'
-- '1326287715'
-- '1386622603'
-- '1477706224'
-- '1518023167'
-- '1922317759'
-- '1205148681'
-- '1255345559'
-- '1497921027'
-- '1760644173'
-- '1952940603'
-- '1659345726'
-- '1699065722'
-- '1336670355'
-- '1376648584'
-- '1457557704'
-- '1134562143'
-- '1396979639'
-- '1558666032'
-- '1386623940'
-- '1093392748'
-- '1144407958'
-- '1295730968'
-- '1861729048'
-- '1336191592'
-- '1720249584'
-- '1023317674'
-- '1356320311'
-- '1003005745'
-- '1285788489'
-- '1720027618'
-- '1205157476'
-- '1326062308'
-- '1578532255'
-- '1710164827'
-- '1881804672'
-- '1922140946'
-- '1093991499'
-- '1285064501'
-- '1376612465'
-- '1780830083'
-- '1164719894'
-- '1659581262'
-- '1689747362'
-- '1063421808'
-- '1447338058'
-- '1538134762'
-- '1679665673'
-- '1871502310'
-- '1093037103'
-- '1912964925'
-- '1225408123'
-- '1992311567'
-- '1184829855'
-- '1326306721'
-- '1649318577'
-- '1669455911'
-- '1780660324'
-- '1144328360'
-- '1710201603'
-- '1902260243'
-- '1922064542'
-- '1194144352'
-- '1760844492'
-- '1932633799'
-- '1760691950'
-- '1932184058'
-- '1558871608'
-- '1649536939'
-- '1598863797'
-- '1780868786'
-- '1043628886'
-- '1053387894'
-- '1306034749'
-- '1922219922'
-- '1154634566'
-- '1326207317'
-- '1275611998'
-- '1700908027'
-- '1720344799'
-- '1922041854'
-- '1528037470'
-- '1396762803'
-- '1437140894'
-- '1962455162'
-- '1407084981'
-- '1841663242'
-- '1124117361'
-- '1477865533'
-- '1053322248'
-- '1184628992'
-- '1194796201'
-- '1306874854'
-- '1407847296'
-- '1205092871'
-- '1851658231'
-- '1629422589'
-- '1104110832'
-- '1457671935'
-- '1497111785'
-- '1972047728'
-- '1285899427'
-- '1376507137'
-- '1457693756'
-- '1538458583'
-- '1609188747'
-- '1942697826'
-- '1548262967'
-- '1063702132'
-- '1144220120'
-- '1194963959'
-- '1447663562'
-- '1649225129'
-- '1083148712'
-- '1144305491'
-- '1154633188'
-- '1386008068'
-- '1366544801'
-- '1952575854'
-- '1952753766'
-- '1912185398'
-- '1255653812'
-- '1295790798'
-- '1891906855'
-- '1972265635'
-- '1487765319'
-- '1720346042'
-- '1376791673'
-- '1649739558'
-- '1861436701'
-- '1114993474'
-- '1124142500'
-- '1497118004'
-- '1497158406'
-- '1285959916'
-- '1346773371'
-- '1801323605'
-- '1790927598'
-- '1992070130'
-- '1427254291'
-- '1215483169'
-- '1487996799'
-- '1760707319'
-- '1801472055'
-- '1134104615'
-- '1427079292'
-- '1730197187'
-- '1770526576'
-- '1508205154'
-- '1568756468'
-- '1649578550'
-- '1962498006'
-- '1982707097'
-- '1194119123'
-- '1811191836'
-- '1497175178'
-- '1588866578'
-- '1639680614'
-- '1710037742'
-- '1043694532'
-- '1407316714'
-- '1730523705'
-- '1760437891'
-- '1932193083'
-- '1952336950'
-- '1922099621'
-- '1023452885'
-- '1053321679'
-- '1396774840'
-- '1780047191'
-- '1932305000'
-- '1295714871'
-- '1841389566'
-- '1962815118'
-- '1073555660'
-- '1235519539'
-- '1851350391'
-- '1982920831'
-- '1265468334'
-- '1922284645'
-- '1992988745'
-- '1295793800'
-- '1215337985'
-- '1467999672'
-- '1639294358'
-- '1144003567'
-- '1679665459'
-- '1952480196'
-- '1669798534'
-- '1134513419'
-- '1356682066'
-- '1437360310'
-- '1144608555'
-- '1629081245'
-- '1932472727'
-- '1487700738'
-- '1922115609'
-- '1922450758'
-- '1730206814'
-- '1689792194'
-- '1902151152'
-- '1023302882'
-- '1942659495'
-- '1982927729'
-- '1992904197'
-- '1932348059'
-- '1164134904'
-- '1992044663'
-- '1194010777'
-- '1407943673'
-- '1144296344'
-- '1881985349'
-- '1346279619'
-- '1629000963'
-- '1881796118'
-- '1184929101'
-- '1295875896'
-- '1114365509'
-- '1336452960'
-- '1508950874'
-- '1891872008'
-- '1124087101'
-- '1639601966'
-- '1336287374'
-- '1356370092'
-- '1396775243'
-- '1588820021'
-- '1659679934'
-- '1710407705'
-- '1891760708'
-- '1942719273'
-- '1295728848'
-- '1538128137'
-- '1710146543'
-- '1932351897'
-- '1619505401'
-- '1902009418'
-- '1477520294'
-- '1699062158'
-- '1043205966'
-- '1992892624'
-- '1700153491'
-- '1720054737'
-- '1801555818'
-- '1174917280'
-- '1427144989'
-- '1215376868'
-- '1225275654'
-- '1619022035'
-- '1447455357'
-- '1114999125'
-- '1396118667'
-- '1477971687'
-- '1760692917'
-- '1972668796'
-- '1780144055'
-- '1215460571'
-- '1710947759'
-- '1861835340'
-- '1316184245'
-- '1326667106'
-- '1962815795'
-- '1255751012'
-- '1346268885'
-- '1881624831'
-- '1073669297'
-- '1124237920'
-- '1649386905'
-- '1801005251'
-- '1093927329'
-- '1437449089'
-- '1528070844'
-- '1588051171'
-- '1649510017'
-- '1669720926'
-- '1114574290'
-- '1659645612'
-- '1821372228'
-- '1982754925'
-- '1346628617'
-- '1124293469'
-- '1184903106'
-- '1285726026'
-- '1194373910'
-- '1801988928'
-- '1437474921'
-- '1467521997'
-- '1154581874'
-- '1336259670'
-- '1760700108'
-- '1841432598'
-- '1881615599'
-- '1194197186'
-- '1912027798'
-- '1144568007'
-- '1316013709'
-- '1508946864'
-- '1669844510'
-- '1740626969'
-- '1891995122'
-- '1558381764'
-- '1902067796'
-- '1235144742'
-- '1407802267'
-- '1295995751'
-- '1205952306'
-- '1386082014'
-- '1477648830'
-- '1538105788'
-- '1568690907'
-- '1265541262'
-- '1730332495'
-- '1124072095'
-- '1336719921'
-- '1396132239'
-- '1104212075'
-- '1164497483'
-- '1083914600'
-- '1548466063'
-- '1275799025'
-- '1790566867'
-- '1801237037'
-- '1861636276'
-- '1598845927'
-- '1982712006'
-- '1053311779'
-- '1811410723'
-- '1598769259'
-- '1649432857'
-- '1285775262'
-- '1154533800'
-- '1992149959'
-- '1497173892'
-- '1841904133'
-- '1841231594'
-- '1851306450'
-- '1881707479'
-- '1013972595'
-- '1639581804'
-- '1720212715'
-- '1073718946'
-- '1205271129'
-- '1396155446'
-- '1659633725'
-- '1760794424'
-- '1184102766'
-- '1700863644'
-- '1790195956'
-- '1033172952'
-- '1679561021'
-- '1093827255'
-- '1235259912'
-- '1316329808'
-- '1386622256'
-- '1619169737'
-- '1295120657'
-- '1649430364'
-- '1104844570'
-- '1063157857'
-- '1437193117'
-- '1548647373'
-- '1063510881'
-- '1396776159'
-- '1336119510'
-- '1568880227'
-- '1023522281'
-- '1336148865'
-- '1649561382'
-- '1811081458'
-- '1235556416'
-- '1740486380'
-- '1073975504'
-- '1417151002'
-- '1215997192'
-- '1740251347'
-- '1083086375'
-- '1497081830'
-- '1538509807'
-- '1821022203'
-- '1275544207'
-- '1528519725'
-- '1992897789'
-- '1316916711'
-- '1720619117'
-- '1558725135'
-- '1740434398'
-- '1760884860'
-- '1831392885'
-- '1386934388'
-- '1194110999'
-- '1326018813'
-- '1396812749'
-- '1558324640'
-- '1700934684'
-- '1952539371'
-- '1265284004'
-- '1083857858'
-- '1144832064'
-- '1629489067'
-- '1932587987'
-- '1205868155'
-- '1417390667'
-- '1629150370'
-- '1245427889'
-- '1861587602'
-- '1356631790'
-- '1518209857'
-- '1689744328'
-- '1497729628'
-- '1528048980'
-- '1659660686'
-- '1063585255'
-- '1841395423'
-- '1922325844'
-- '1669852661'
-- '1750554747'
-- '1083902761'
-- '1275293524'
-- '1639675069'
-- '1982865705'
-- '1558708800'
-- '1457648917'
-- '1841452885'
-- '1043277924'
-- '1801138573'
-- '1871540518'
-- '1154428928'
-- '1013270370'
-- '1407933187'
-- '1730124785'
-- '1548363724'
-- '1699138685'
-- '1396784518'
-- '1841482387'
-- '1861628828'
-- '1952680944'
-- '1003846361'
-- '1295938918'
-- '1932365509'
-- '1235579665'
-- '1659533016'
-- '1730262510'
-- '1982877445'
-- '1417479544'
-- '1013970169'
-- '1841401148'
-- '1902989783'
-- '1295922425'
-- '1477896462'
-- '1013935436'
-- '1033457270'
-- '1053417220'
-- '1134160385'
-- '1356942460'
-- '1235348855'
-- '1235593039'
-- '1699893362'
-- '1154401982'
-- '1497188635'
-- '1720164080'
-- '1063702314'
-- '1598227746'
-- '1831212505'
-- '1891953659'
-- '1083794341'
-- '1386790996'
-- '1497752810'
-- '1639490857'
-- '1447366794'
-- '1477955540'
-- '1558660886'
-- '1821388844'
-- '1861987570'
-- '1205876117'
-- '1619948593'
-- '1083642565'
-- '1205891470'
-- '1245684562'
-- '1750317053'
-- '1881701571'
-- '1063403384'
-- '1336301977'
-- '1376556035'
-- '1528036415'
-- '1548251630'
-- '1760943625'
-- '1992971550'
-- '1427291095'
-- '1952327926'
-- '1225292782'
-- '1174835631'
-- '1255507216'
-- '1174818363'
-- '1326575846'
-- '1063678852'
-- '1598030827'
-- '1952729048'
-- '1427248756'
-- '1073574364'
-- '1881651396'
-- '1073565396'
-- '1205884830'
-- '1356561229'
-- '1891893814'
-- '1932301512'
-- '1972888774'
-- '1245226893'
-- '1730724980'
-- '1760772412'
-- '1942597356'
-- '1285851436'
-- '1811376593'
-- '1891984316'
-- '1013019330'
-- '1154414225'
-- '1407143878'
-- '1285600221'
-- '1720107634'
-- '1770557977'
-- '1437433471'
-- '1467498980'
-- '1518128974'
-- '1730244310'
-- '1013520832'
-- '1134319619'
-- '1457636235'
-- '1679537369'
-- '1699188623'
-- '1437371127'
-- '1700201969'
-- '1912427071'
-- '1699978916'
-- '1013940014'
-- '1730682576'
-- '1316103997'
-- '1790987477'
-- '1376950477'
-- '1558937078'
-- '1821229170'
-- '1811489016'
-- '1518352574'
-- '1790458990'
-- '1265607758'
-- '1538378724'
-- '1598831927'
-- '1942220710'
-- '1265872519'
-- '1437672623'
-- '1023207230'
-- '1427265461'
-- '1467410126'
-- '1477973139'
-- '1235303678'
-- '1639106503'
-- '1003962614'
-- '1528033990'
-- '1811067556'
-- '1235189697'
-- '1841617396'
-- '1972697357'
-- '1447691977'
-- '1689669699'
-- '1063750479'
-- '1396702015'
-- '1043327752'
-- '1225246754'
-- '1225396930'
-- '1760803647'
-- '1326395096'
-- '1124319462'
-- '1235290693'
-- '1245646702'
-- '1447494208'
-- '1780170209'
-- '1982790259'
-- '1124360953'
-- '1245269521'
-- '1124033634'
-- '1831145689'
-- '1477624237'
-- '1699716019'
-- '1720407133'
-- '1952493827'
-- '1871002857'
-- '1326017690'
-- '1619949716'
-- '1649479411'
-- '1730749870'
-- '1144577834'
-- '1790704955'
-- '1912984592'
-- '1437172640'
-- '1093124836'
-- '1356496806'
-- '1619957750'
-- '1194805598'
-- '1497174544'
-- '1801334404'
-- '1982768396'
-- '1053394098'
-- '1275889792'
-- '1023433299'
-- '1811128655'
-- '1134387665'
-- '1255398301'
-- '1609274083'
-- '1619113552'
-- '1790733194'
-- '1295815751'
-- '1497813448'
-- '1699035030'
-- '1871755884'
-- '1285271429'
-- '1992737423'
-- '1093377764'
-- '1396060554'
-- '1437344942'
-- '1053405068'
-- '1235230707'
-- '1356535017'
-- '1396950978'
-- '1043238348'
-- '1366400756'
-- '1518533207'
-- '1710974100'
-- '1851513428'
-- '1649462425'
-- '1811317290'
-- '1114008349'
-- '1841241627'
-- '1942520515'
-- '1033139472'
-- '1740445311'
-- '1053523399'
-- '1184615825'
-- '1578726907'
-- '1831168244'
-- '1194764670'
-- '1053722645'
-- '1225207194'
-- '1295099240'
-- '1790970986'
-- '1134440399'
-- '1316380249'
-- '1124400684'
-- '1316262710'
-- '1740208750'
-- '1932463825'
-- '1154579175'
-- '1356339170'
-- '1841862844'
-- '1033409479'
-- '1801982434'
-- '1235583808'
-- '1366512451'
-- '1306872270'
-- '1366741548'
-- '1841648557'
-- '1952409005'
-- '1114108719'
-- '1346632551'
-- '1457988305'
-- '1235153743'
-- '1306008339'
-- '1427505890'
-- '1437595154'
-- '1740771617'
-- '1932317906'
-- '1407376122'
-- '1508849308'
-- '1518047943'
-- '1801280250'
-- '1750690814'
-- '1376562017'
-- '1578750949'
-- '1982795233'
-- '1154758845'
-- '1164788741'
-- '1124097381'
-- '1184858243'
-- '1700103041'
-- '1861492209'
-- '1669533766'
-- '1821027483'
-- '1891088613'
-- '1093974420'
-- '1164447355'
-- '1811264146'
-- '1821433665'
-- '1871798249'
-- '1407909344'
-- '1740422898'
-- '1306239298'
-- '1003298126'
-- '1023309630'
-- '1548585342'
-- '1922380930'
-- '1316111719'
-- '1477524452'
-- '1487844403'
-- '1134117096'
-- '1245549351'
-- '1720378904'
-- '1831364058'
-- '1164516944'
-- '1699745737'
-- '1710937982'
-- '1710954409'
-- '1043298896'
-- '1710272034'
-- '1437225737'
-- '1487664009'
-- '1710331723'
-- '1912014770'
-- '1295137743'
-- '1861887507'
-- '1649284910'
-- '1730634759'
-- '1811247976'
-- '1285925305'
-- '1972703676'
-- '1275582611'
-- '1851755458'
-- '1548648116'
-- '1699986075'
-- '1467577254'
-- '1033667548'
-- '1073779641'
-- '1114025301'
-- '1356683528'
-- '1992150999'
-- '1275797656'
-- '1487068623'
-- '1710377957'
-- '1174792915'
-- '1104908011'
-- '1790941367'
-- '1740282623'
-- '1043303050'
-- '1144273178'
-- '1225491764'
-- '1831306851'
-- '1760793046'
-- '1861930042'
-- '1942663802'
-- '1447506217'
-- '1730185406'
-- '1861782484'
-- '1639306020'
-- '1669508073'
-- '1821437757'
-- '1861759193'
-- '1437183316'
-- '1982658050'
-- '1215015888'
-- '1649601626'
-- '1760626352'
-- '1154493476'
-- '1265645626'
-- '1073915427'
-- '1548216690'
-- '1902830532'
-- '1043423247'
-- '1285619726'
-- '1356364830'
-- '1356402218'
-- '1760952394'
-- '1942656517'
-- '1801003793'
-- '1710904164'
-- '1932191145'
-- '1578687109'
-- '1134307713'
-- '1508975376'
-- '1548657182'
-- '1801071691'
-- '1750678900'
-- '1760793822'
-- '1467896217'
-- '1497845762'
-- '1851446330'
-- '1245253772'
-- '1235556192'
-- '1295051290'
-- '1992886949'
-- '1740202027'
-- '1902857451'
-- '1275149189'
-- '1649258484'
-- '1972508901'
-- '1083317218'
-- '1003802166'
-- '1376034876'
-- '1821184078'
-- '1487801288'
-- '1669599395'
-- '1710299078'
-- '1497758643'
-- '1952392888'
-- '1912964941'
-- '1053648097'
-- '1881658300'
-- '1902125057'
-- '1275783284'
-- '1023213030'
-- '1093706988'
-- '1215107636'
-- '1710390547'
-- '1083649388'
-- '1104112143'
-- '1366004988'
-- '1972779775'
-- '1386835569'
-- '1356720098'
-- '1720200918'
-- '1578614723'
-- '1629134697'
-- '1053973248'
-- '1194787358'
-- '1477724862'
-- '1003829771'
-- '1952337610'
-- '1295835593'
-- '1477661908'
-- '1689960007'
-- '1932192200'
-- '1972827772'
-- '1023136439'
-- '1083794259'
-- '1417619172'
-- '1528226784'
-- '1780670331'
-- '1780833483'
-- '1134473978'
-- '1134562614'
-- '1356916894'
-- '1952397473'
-- '1225018120'
-- '1447548193'
-- '1255870978'
-- '1639193873'
-- '1184974230'
-- '1104144690'
-- '1124291075'
-- '1336448026'
-- '1538484829'
-- '1871923201'
-- '1699971721'
-- '1407965494'
-- '1528035151'
-- '1649552688'
-- '1811242704'
-- '1124439971'
-- '1407051527'
-- '1336305275'
-- '1558524629'
-- '1649291048'
-- '1053673863'
-- '1124117254'
-- '1295913523'
-- '1407989585'
-- '1659639151'
-- '1093721623'
-- '1174817456'
-- '1326334236'
-- '1861481343'
-- '1437425022'
-- '1578926606'
-- '1184677296'
-- '1285770925'
-- '1285908962'
-- '1568728244'
-- '1043535503'
-- '1053378240'
-- '1417340035'
-- '1821026964'
-- '1346289030'
-- '1316973464'
-- '1174541858'
-- '1619107349'
-- '1891891362'
-- '1083683676'
-- '1326308966'
-- '1326475633'
-- '1366468969'
-- '1942543962'
-- '1992163281'
-- '1487819785'
-- '1164614707'
-- '1427167287'
-- '1811949597'
-- '1891723326'
-- '1164650644'
-- '1265625701'
-- '1336650456'
-- '1689863631'
-- '1871553115'
-- '1871154914'
-- '1649657461'
-- '1386006823'
-- '1881056950'
-- '1083108054'
-- '1154539385'
-- '1497792667'
-- '1306856927'
-- '1023216892'
-- '1073004578'
-- '1245252790'
-- '1295091510'
-- '1336152628'
-- '1750890810'
-- '1154907830'
-- '1548587033'
-- '1558302372'
-- '1891718243'
-- '1396790788'
-- '1467427179'
-- '1952325318'
-- '1306994595'
-- '1497152904'
-- '1194915306'
-- '1598792079'
-- '1316384555'
-- '1154741213'
-- '1801530985'
-- '1417117243'
-- '1790153518'
-- '1003971888'
-- '1063943405'
-- '1689863052'
-- '1265744783'
-- '1417217530'
-- '1013923283'
-- '1104090273'
-- '1306826896'
-- '1568708782'
-- '1760693063'
-- '1952344517'
-- '1356597561'
-- '1417994476'
-- '1720133945'
-- '1093747156'
-- '1336274182'
-- '1649326232'
-- '1497011472'
-- '1083054670'
-- '1003906504'
-- '1396032660'
-- '1033138243'
-- '1043482698'
-- '1801810353'
-- '1033215728'
-- '1124436753'
-- '1063851186'
-- '1174951370'
-- '1285980193'
-- '1508063942'
-- '1700142551'
-- '1053916254'
-- '1982920765'
-- '1063562775'
-- '1164518494'
-- '1619055209'
-- '1346938511'
-- '1003006214'
-- '1275053696'
-- '1518157510'
-- '1134431968'
-- '1598845406'
-- '1134534613'
-- '1467744185'
-- '1518320381'
-- '1861582348'
-- '1639365802'
-- '1962470849'
-- '1992836027'
-- '1245416197'
-- '1699112607'
-- '1467838201'
-- '1780003426'
-- '1295827939'
-- '1619088580'
-- '1972506749'
-- '1144471681'
-- '1144271651'
-- '1669892295'
-- '1942628573'
-- '1093118622'
-- '1063460558'
-- '1487065603'
-- '1750338026'
-- '1720150568'
-- '1740300482'
-- '1740597111'
-- '1104212653'
-- '1790992832'
-- '1285794529'
-- '1356410625'
-- '1558591867'
-- '1972820918'
-- '1013232735'
-- '1558453258'
-- '1619140100'
-- '1760703482'
-- '1922057116'
-- '1144732850'
-- '1518118033'
-- '1558650614'
-- '1700106556'
-- '1710915236'
-- '1962466375'
-- '1912240367'
-- '1174752067'
-- '1831131341'
-- '1548673841'
-- '1972821726'
-- '1437332541'
-- '1841264991'
-- '1134789530'
-- '1073046801'
-- '1104270644'
-- '1053848275'
-- '1902194939'
-- '1376583807'
-- '1740263474'
-- '1396974259'
-- '1740218296'
-- '1992779334'
-- '1457377681'
-- '1376563841'
-- '1750379459'
-- '1346222197'
-- '1477654051'
-- '1568548543'
-- '1861562985'
-- '1558477141'
-- '1659442036'
-- '1114192093'
-- '1265870950'
-- '1043553589'
-- '1265776587'
-- '1417987553'
-- '1538253570'
-- '1275585945'
-- '1932408119'
-- '1720058092'
-- '1033315247'
-- '1962764860'
-- '1508066309'
-- '1700846649'
-- '1871734509'
-- '1265455968'
-- '1700177086'
-- '1962490599'
-- '1205947785'
-- '1669494100'
-- '1881089688'
-- '1629362876'
-- '1568492981'
-- '1508155391'
-- '1760505622'
-- '1023030657'
-- '1265560437'
-- '1558310599'
-- '1427263235'
-- '1861609034'
-- '1932366549'
-- '1710196019'
-- '1942662994'
-- '1245360478'
-- '1942558010'
-- '1629411798'
-- '1053528075'
-- '1396916524'
-- '1487072765'
-- '1679599575'
-- '1376629055'
-- '1093414864'
-- '1972835023'
-- '1164659637'
-- '1003200262'
-- '1831166990'
-- '1124364708'
-- '1518048743'
-- '1992906135'
-- '1184649154'
-- '1467563551'
-- '1134231293'
-- '1144300237'
-- '1326313453'
-- '1457525230'
-- '1336671866'
-- '1689933020'
-- '1972504314'
-- '1477746980'
-- '1932291077'
-- '1861866717'
-- '1114958402'
-- '1790001923'
-- '1003264458'
-- '1285071118'
-- '1639463367'
-- '1477954865'
-- '1477051522'
-- '1982784245'
-- '1699875716'
-- '1578727749'
-- '1265778831'
-- '1891767307'
-- '1598337594'
-- '1982869079'
-- '1194812263'
-- '1962889667'
-- '1477715696'
-- '1407200272'
-- '1275511453'
-- '1700848868'
-- '1275909301'
-- '1386952950'
-- '1639600257'
-- '1851824510'
-- '1841486974'
-- '1467642603'
-- '1548551385'
-- '1134430994'
-- '1477909901'
-- '1477949873'
-- '1053567982'
-- '1366705675'
-- '1104185354'
-- '1194987081'
-- '1699090068'
-- '1073635918'
-- '1497020648'
-- '1518998871'
-- '1750812228'
-- '1821331216'
-- '1992190599'
-- '1588624431'
-- '1154423572'
-- '1396130407'
-- '1780814137'
-- '1669477493'
-- '1124376231'
-- '1457449837'
-- '1326192071'
-- '1336167766'
-- '1023329208'
-- '1134626542'
-- '1134370737'
-- '1831333921'
-- '1558776567'
-- '1306838735'
-- '1730491887'
-- '1457592396'
-- '1780669325'
-- '1487065850'
-- '1548558117'
-- '1417489709'
-- '1508074568'
-- '1528151859'
-- '1609212208'
-- '1457554271'
-- '1992900641'
-- '1134449143'
-- '1073524211'
-- '1689965808'
-- '1760432272'
-- '1770618498'
-- '1689694119'
-- '1780775056'
-- '1144528498'
-- '1912163825'
-- '1588921241'
-- '1215341805'
-- '1255422531'
-- '1043203995'
-- '1164494688'
-- '1134793573'
-- '1295047942'
-- '1063450021'
-- '1821299934'
-- '1649448713'
-- '1942696489'
-- '1376503714'
-- '1033209770'
-- '1326242124'
-- '1336460583'
-- '1346696069'
-- '1396817755'
-- '1639153174'
-- '1558566794'
-- '1265726285'
-- '1518982776'
-- '1548587199'
-- '1417174129'
-- '1215957113'
-- '1992789754'
-- '1760620835'
-- '1124133574'
-- '1427044700'
-- '1831182302'
-- '1770039729'
-- '1245499672'
-- '1255759494'
-- '1518255280'
-- '1477796258'
-- '1124104070'
-- '1164508479'
-- '1619068988'
-- '1578655353'
-- '1942596721'
-- '1740361930'
-- '1790895506'
-- '1235365081'
-- '1841635463'
-- '1245547611'
-- '1649723339'
-- '1417984899'
-- '1164774675'
-- '1891734430'
-- '1912294620'
-- '1306834502'
-- '1194450940'
-- '1346279981'
-- '1497118731'
-- '1174970149'
-- '1598103392'
-- '1316186323'
-- '1376602292'
-- '1578707410'
-- '1740254473'
-- '1114350451'
-- '1588197164'
-- '1770603722'
-- '1104144930'
-- '1134102882'
-- '1134523046'
-- '1033194261'
-- '1598777633'
-- '1083056253'
-- '1568553055'
-- '1750487401'
-- '1891154530'
-- '1699260018'
-- '1548206527'
-- '1013128693'
-- '1215324512'
-- '1861658718'
-- '1700448610'
-- '1285228890'
-- '1497850663'
-- '1811915598'
-- '1639432628'
-- '1407880198'
-- '1740244078'
-- '1548425622'
-- '1376650754'
-- '1730584327'
-- '1104569417'
-- '1700987203'
-- '1942615224'
-- '1780843029'
-- '1356553457'
-- '1912509829'
-- '1851607394'
-- '1629150503'
-- '1780034900'
-- '1245240563'
-- '1851757389'
-- '1235112160'
-- '1215424106'
-- '1528080199'
-- '1689931354'
-- '1013956267'
-- '1386690360'
-- '1760634802'
-- '1285689703'
-- '1043871916'
-- '1932257441'
-- '1275733537'
-- '1750644779'
-- '1003221342'
-- '1972787760'
-- '1386032266'
-- '1477516102'
-- '1710199161'
-- '1609294701'
-- '1134270192'
-- '1205315751'
-- '1073613386'
-- '1447270087'
-- '1528049483'
-- '1699734434'
-- '1245204635'
-- '1023450004'
-- '1396934972'
-- '1316451305'
-- '1730783754'
-- '1275624298'
-- '1447646708'
-- '1952427585'
-- '1588674105'
-- '1255423943'
-- '1407976756'
-- '1013284835'
-- '1386078814'
-- '1568541027'
-- '1790787547'
-- '1023574175'
-- '1831308576'
-- '1912071663'
-- '1134360746'
-- '1265630545'
-- '1477573988'
-- '1659874980'
-- '1992755326'
-- '1750590790'
-- '1639185077'
-- '1194932061'
-- '1023397627'
-- '1528055753'
-- '1639260243'
-- '1366746893'
-- '1114173424'
-- '1700841566'
-- '1164628574'
-- '1992872717'
-- '1831285402'
-- '1780073742'
-- '1174595466'
-- '1073856043'
-- '1316056641'
-- '1013018035'
-- '1457489288'
-- '1114306537'
-- '1457779514'
-- '1619026135'
-- '1538106828'
-- '1306854815'
-- '1457580607'
-- '1063850907'
-- '1770903882'
-- '1376025221'
-- '1710977822'
-- '1215464037'
-- '1437262300'
-- '1609154798'
-- '1720127715'
-- '1821435009'
-- '1124081799'
-- '1043226368'
-- '1558395665'
-- '1629184460'
-- '1871680199'
-- '1871649129'
-- '1720043524'
-- '1801886478'
-- '1619298080'
-- '1205852480'
-- '1700839909'
-- '1851711667'
-- '1316930837'
-- '1437248309'
-- '1376591131'
-- '1669779708'
-- '1437339694'
-- '1497980031'
-- '1043273337'
-- '1700869997'
-- '1508829938'
-- '1538480603'
-- '1881846483'
-- '1467428979'
-- '1669595872'
-- '1972600179'
-- '1790191831'
-- '1619001872'
-- '1407974116'
-- '1407143217'
-- '1386963999'
-- '1376832592'
-- '1013934058'
-- '1316904550'
-- '1134460546'
-- '1922623214'
-- '1306885876'
-- '1588033161'
-- '1245240639'
-- '1588602270'
-- '1891112108'
-- '1346469202'
-- '1083036669'
-- '1770749418'
-- '1417915422'
-- '1710102652'
-- '1093982357'
-- '1609343425'
-- '1558484931'
-- '1326008525'
-- '1821080979'
-- '1114944063'
-- '1336455690'
-- '1104820802'
-- '1124257407'
-- '1396737458'
-- '1073873634'
-- '1124292537'
-- '1023081395'
-- '1134788433'
-- '1134383326'
-- '1265788228'
-- '1497866578'
-- '1134362387'
-- '1053835124'
-- '1487116711'
-- '1295768521'
-- '1427008556'
-- '1912056078'
-- '1386148815'
-- '1063425296'
-- '1285677203'
-- '1285691535'
-- '1598840068'
-- '1164008090'
-- '1912963182'
-- '1043300932'
-- '1760846315'
-- '1225068307'
-- '1699718411'
-- '1477579795'
-- '1508116773'
-- '1881097012'
-- '1932108149'
-- '1508892738'
-- '1386978914'
-- '1619242807'
-- '1295178507'
-- '1053491316'
-- '1306805759'
-- '1831402668'
-- '1851599963'
-- '1548226616'
-- '1093247934'
-- '1992961502'
-- '1316946288'
-- '1720230725'
-- '1396721049'
-- '1710901848'
-- '1831167238'
-- '1295790566'
-- '1952798258'
-- '1952863078'
-- '1124095948'
-- '1033149299'
-- '1740790179'
-- '1033301759'
-- '1083617526'
-- '1144286592'
-- '1902875172'
-- '1205476447'
-- '1811275944'
-- '1508819582'
-- '1669630877'
-- '1720054471'
-- '1669448734'
-- '1669811139'
-- '1154360378'
-- '1134118706'
-- '1629399035'
-- '1972514461'
-- '1154384360'
-- '1861876096'
-- '1831137876'
-- '1205821014'
-- '1265671101'
-- '1649569609'
-- '1962608059'
-- '1144481896'
-- '1942760681'
-- '1770848541'
-- '1598833303'
-- '1881988368'
-- '1922392885'
-- '1407947047'
-- '1356767297'
-- '1003034455'
-- '1407958556'
-- '1730169244'
-- '1427562909'
-- '1417295148'
-- '1922251586'
-- '1265572499'
-- '1720466659'
-- '1801472683'
-- '1750519385'
-- '1528589173'
-- '1164963468'
-- '1356332506'
-- '1528047057'
-- '1134302193'
-- '1255618906'
-- '1366695439'
-- '1194990549'
-- '1053951442'
-- '1548578560'
-- '1447335351'
-- '1295368488'
-- '1982995395'
-- '1952584948'
-- '1669739355'
-- '1326332438'
-- '1942361662'
-- '1922172600'
-- '1073565867'
-- '1659378628'
-- '1194734640'
-- '1740743210'
-- '1386931509'
-- '1639345598'
-- '1093193641'
-- '1457613564'
-- '1174575484'
-- '1780942649'
-- '1568444875'
-- '1972519122'
-- '1457561359'
-- '1902857147'
-- '1083969612'
-- '1528084357'
-- '1871834325'
-- '1063916039'
-- '1871028076'
-- '1538562673'
-- '1407852528'
-- '1164628053'
-- '1710958350'
-- '1437389822'
-- '1508371014'
-- '1750647798'
-- '1689787384'
-- '1841264033'
-- '1063451763'
-- '1477968006'
-- '1396039798'
-- '1083693691'
-- '1629079215'
-- '1700954641'
-- '1629082607'
-- '1033130885'
-- '1871591578'
-- '1760846281'
-- '1598072589'
-- '1710245055'
-- '1447584222'
-- '1619139128'
-- '1558335083'
-- '1295749422'
-- '1669017935'
-- '1518599646'
-- '1588641740'
-- '1649208521'
-- '1912909615'
-- '1508245572'
-- '1306023593'
-- '1285697490'
-- '1750551420'
-- '1386873677'
-- '1942430715'
-- '1689948150'
-- '1619185675'
-- '1356337273'
-- '1780641555'
-- '1780618462'
-- '1467689034'
-- '1992756373'
-- '1336194125'
-- '1083870919'
-- '1821268293'
-- '1346284957'
-- '1720335045'
-- '1700835832'
-- '1942590484'
-- '1063556991'
-- '1245472349'
-- '1255421848'
-- '1245297480'
-- '1639326275'
-- '1487816443'
-- '1902986243'
-- '1508011776'
-- '1205002797'
-- '1154725844'
-- '1649224502'
-- '1497000566'
-- '1982809315'
-- '1326178302'
-- '1811464621'
-- '1033177928'
-- '1295940336'
-- '1093776585'
-- '1780669390'
-- '1346298684'
-- '1265634562'
-- '1437123247'
-- '1154339570'
-- '1356444749'
-- '1407944739'
-- '1881617835'
-- '1316901549'
-- '1609119015'
-- '1336350891'
-- '1861417271'
-- '1801184973'
-- '1063505030'
-- '1013954619'
-- '1457573271'
-- '1114179991'
-- '1235565409'
-- '1578924247'
-- '1124013826'
-- '1629363163'
-- '1871703678'
-- '1386955201'
-- '1366477481'
-- '1952515538'
-- '1366732893'
-- '1801060462'
-- '1255534855'
-- '1770546855'
-- '1093776668'
-- '1669890745'
-- '1033384805'
-- '1609839844'
-- '1255381794'
-- '1932149499'
-- '1588677439'
-- '1376579870'
-- '1770542359'
-- '1114240561'
-- '1063407575'
-- '1922172162'
-- '1609854538'
-- '1114319324'
-- '1891792040'
-- '1760491781'
-- '1902163926'
-- '1619963246'
-- '1154536308'
-- '1255687034'
-- '1083895338'
-- '1386890135'
-- '1376646695'
-- '1033475793'
-- '1568648061'
-- '1013194513'
-- '1831499409'
-- '1447224571'
-- '1710971478'
-- '1609099431'
-- '1821016502'
-- '1710936448'
-- '1669423562'
-- '1821038548'
-- '1336240332'
-- '1124342662'
-- '1528453313'
-- '1821165762'
-- '1790705861'
-- '1326363318'
-- '1356379556'
-- '1962413492'
-- '1992037931'
-- '1275617375'
-- '1962744797'
-- '1386664019'
-- '1558527762'
-- '1003903493'
-- '1265962559'
-- '1255436697'
-- '1598923096'
-- '1053514844'
-- '1568492940'
-- '1144230962'
-- '1669636536'
-- '1558492199'
-- '1104881697'
-- '1194843722'
-- '1821282484'
-- '1831489053'
-- '1376553255'
-- '1073763140'
-- '1831416882'
-- '1740424621'
-- '1548436447'
-- '1245399807'
-- '1114992203'
-- '1295907194'
-- '1801963749'
-- '1003080078'
-- '1275732372'
-- '1851548960'
-- '1871895573'
-- '1346623840'
-- '1750554952'
-- '1487621199'
-- '1194046318'
-- '1992890883'
-- '1407860331'
-- '1619135845'
-- '1760640403'
-- '1699861047'
-- '1952422719'
-- '1326144486'
-- '1154329209'
-- '1316172703'
-- '1518932250'
-- '1467612382'
-- '1841630142'
-- '1861694887'
-- '1700102597'
-- '1710393996'
-- '1346425584'
-- '1558521013'
-- '1669537478'
-- '1902833684'
-- '1730220666'
-- '1326210311'
-- '1891885554'
-- '1235492547'
-- '1891945895'
-- '1104913045'
-- '1669450094'
-- '1255415360'
-- '1407108913'
-- '1184633059'
-- '1760485890'
-- '1013993740'
-- '1083698021'
-- '1154409423'
-- '1265494835'
-- '1851357388'
-- '1700317286'
-- '1114976214'
-- '1013119320'
-- '1518050608'
-- '1396906368'
-- '1982654448'
-- '1154318970'
-- '1649211798'
-- '1609162387'
-- '1609219989'
-- '1841580628'
-- '1467537415'
-- '1790757557'
-- '1760571467'
-- '1619066149'
-- '1508007675'
-- '1093815904'
-- '1962418145'
-- '1811961162'
-- '1396787602'
-- '1093809659'
-- '1760707046'
-- '1548292402'
-- '1477796068'
-- '1336140441'
-- '1699917104'
-- '1043224140'
-- '1912112020'
-- '1194906321'
-- '1760775381'
-- '1831131606'
-- '1699934091'
-- '1669682837'
-- '1184607525'
-- '1306815014'
-- '1245527787'
-- '1831109461'
-- '1962602904'
-- '1427076454'
-- '1326243320'
-- '1467512871'
-- '1093741175'
-- '1629426085'
-- '1083730105'
-- '1548211642'
-- '1649251422'
-- '1588778955'
-- '1114188422'
-- '1699950899'
-- '1366538811'
-- '1932555109'
-- '1659344463'
-- '1255480042'
-- '1881868511'
-- '1992931182'
-- '1598077430'
-- '1851341119'
-- '1326091562'
-- '1447281969'
-- '1396766093'
-- '1689764201'
-- '1831223049'
-- '1295793172'
-- '1407828411'
-- '1508055468'
-- '1740246743'
-- '1578705612'
-- '1679951347'
-- '1932423530'
-- '1477697068'
-- '1407177124'
-- '1467465971'
-- '1154529774'
-- '1477744258'
-- '1245321595'
-- '1477701654'
-- '1538185756'
-- '1962442079'
-- '1366426140'
-- '1952420705'
-- '1033313978'
-- '1336179662'
-- '1760642656'
-- '1821248451'
-- '1497932024'
-- '1386662062'
-- )

In [0]:
select distinct call_channel__v from com_edp_prd.com_raw.vcrm_call2__v

In [0]:
select state, count(distinct territory_name)
from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
group by 1 order by 2 desc

In [0]:
with all_claims as (
  SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Dx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Dx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761','E763')
  AND TRANSACTION_STATUS = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001','540920700')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Tx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001','540920700')
  AND TRANSACTION_RESULT = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                          '38206','38230','38232','38240','38241','38242','38243','38250')
),
relevant_patients as (
  select *
  from all_claims
  where patient_id in (select distinct patient_id from com_edp_prd.cmpa_insights_internal_schema.patient360_master)
),
patient_geography AS (
  SELECT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),
pulling_relevant_info as (
  select a.patient_id, a.npi, a.fill_date, b.PATIENT_YOB, (year(current_date) - year(b.PATIENT_YOB)) as age, c.PATIENT_STATE, d.FIRST_NAME, d.LAST_NAME, concat(d.FIRST_NAME, ' ', d.LAST_NAME) as hcp_name, d.PRIMARY_SPECIALTY, d.SECONDARY_SPECIALTY, a.claim_type, e.PAYER_NAME, e.INSURANCE_GROUP
  from relevant_patients as a
  left join com_edp_prd.com_raw.kom_patient_demographics as b on a.patient_id = b.PATIENT_ID
  left join patient_geography as c on a.patient_id = c.PATIENT_ID
  left join com_edp_prd.com_raw.kom_providers as d on a.npi = d.npi and d.provider_type = 'INDIVIDUAL'
  left join com_edp_prd.com_raw.kom_plans as e on a.plan_id = e.KH_PLAN_ID
)
select * from pulling_relevant_info

### Payer 360 Analysis

In [0]:
with t1 as (select MEDICAL_EVENT_ID, coalesce(RENDERING_NPI, REFERRING_NPI) as npi
from com_edp_prd.com_raw.kom_medical_events)

select count(distinct medical_event_id)
from t1
where npi is null

In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

In [0]:
WITH
/* ============================================================================
   1) ELIGIBILITY COHORT BUILD
   Goal: Identify eligible MPS II patients using:
     A) "Specified" Dx (E761) with >=2 distinct Dx dates + ANY qualifying treatment evidence
     B) "Incremental Unspecified" Dx (E763) with >=2 distinct Dx dates + Elaprase-only evidence
        and NOT already included in (A)
   ========================================================================== */

-- Pull all "Specified" diagnosis events (E761) within 5-year-ish window for Dx counting.
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),

-- Keep patients with >=2 distinct Dx dates for "Specified".
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Pull all "Unspecified" diagnosis events (E763) within the same window for Dx counting.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),

-- Keep patients with >=2 distinct Dx dates for "Unspecified".
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Treatment evidence universe (broad): Elaprase NDCs OR relevant infusion/procedure codes.
-- Used to ensure "Specified" cohort has some treatment evidence in the more recent window.
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),

-- Treatment evidence (narrow): Elaprase only (NDCs + J1743).
-- Used for incremental inclusion of "Unspecified" cohort.
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),

-- Eligible "Specified" = >=2 Dx dates AND any treatment evidence.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Eligible "Incremental Unspecified" = >=2 Dx dates AND Elaprase-only evidence,
-- excluding anyone already in the specified+treatment set.
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible patient list.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

/* ============================================================================
   2) PROVIDER FILTER ("COHORT 3 LEARNINGS")
   Goal: constrain HCPs to relevant specialties and exclude noise specialties.
   Used to filter Dx/Tx claim NPIs (but still allow NULL NPI claims through).
   ========================================================================== */
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

/* ============================================================================
   3) CLAIMS UNIVERSES (DX + TX) WITH NPI ATTRIBUTION
   - 5Y-ish window (2020-08-01 -> end_date) used for "stats" and "latest"
   - 3Y-ish window (2022-08-01 -> end_date) used for ranking "most-seen"
   ========================================================================== */

-- All diagnosis claims (E761/E763) in the 5Y window, with NPI attribution.
-- Medical uses rendering/referring; pharmacy uses prescriber.
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, MEDICAL_EVENT_ID as claim_id, BILLING_NPI as billing_npi
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, PHARMACY_EVENT_ID as claim_id, PHARMACY_NPI as billing_npi
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- Filter to "allowed" NPIs, but keep NULL NPI rows so patient-level dates won't be lost.
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- All treatment claims in the 5Y window, with a unified TX_CODE field:
--   - Elaprase NDCs from medical/pharmacy
--   - Infusion/procedure codes from medical (TX_CODE = PROCEDURE_CODE)
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE, MEDICAL_EVENT_ID as claim_id, BILLING_NPI as billing_npi
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE, PHARMACY_EVENT_ID as claim_id, PHARMACY_NPI as billing_npi
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE, MEDICAL_EVENT_ID as claim_id, BILLING_NPI as billing_npi
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- NEW: ALL-TIME treatment universe (no date restriction) using same tx definition as above.
-- Used ONLY to compute "first_tx_after_diagnosis" without restricting to the 5Y window.
all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Normalize Dx + Tx into a single 5Y claim stream (TX_CODE NULL for Dx rows).
-- This enables unified "visit count" and "latest claim" logic.
all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE, claim_id, billing_npi
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE, claim_id, billing_npi
    FROM all_tx_claims_5yr
)

select count(distinct claim_id)
from all_claims_5yr
where billing_npi is not null